*** Conformal Prediction & ML-Models ***

*** The following pipeline will implement the following models:
 
1) Decision Trees
2) Random Forest
3) LightBGM
4) XGBoost

The following Conformal Prediction frameworks will be applied as a layer on top of the forecasting models:

1) Split/Inductive Conformal Prediction (baseline)
2) Ensemble Batch Prediction Intervals (EnbPI)
3) Adaptive Conformal Inference (ACI)

In [ ]:
# load in data

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

df = pd.read_csv('brent_exogenoustechnical_stress2.csv')

df.head(5)


In [ ]:
# As we predict t+1 one-step-ahead, we have to shift the target variable by one day backwards
df['brent_price_t1'] = df['brent_price'].shift(-1)

In [ ]:
# rearrange for better readability
cols = df.columns.tolist()

cols.remove('brent_price_t1')
insert_at = cols.index('brent_price') + 1
cols.insert(insert_at, 'brent_price_t1')

df = df[cols]
df.head(5)

In [ ]:
# dropp 1 nan in brent_price_t1
df = df.dropna(subset=['brent_price_t1'])

In [ ]:
# dtypes
print(df.dtypes)

In [ ]:
# date to datetime
df['date'] = pd.to_datetime(df['date'])

In [ ]:
# overview of stress periods already flagged in stress_m10

stress = df["stress_m10"] == 1
period_id = (stress != stress.shift()).cumsum() # create a period id for consecutive stress periods

stress_overview = (
    df.loc[stress, ["date"]]
    .assign(period_id=period_id[stress].values) # assign the period id to stress rows
    .groupby("period_id")
    .agg( # summarzie
        start_date=("date", "min"),
        end_date=("date", "max"),
        length_days=("date", "size")
    )
    .reset_index(drop=True) 
)
stress_overview

# Model Training 

## 1) Decision Trees

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.tree import DecisionTreeRegressor

# define features and target variable
X = df.drop(columns=['brent_price', 'brent_price_t1', 'date', 'stress_m10'])
y = df['brent_price_t1']

# splits
n = len(X)

train_end = int(n * 0.7) # 70% for training
calib_end = int(n * 0.85) # 15% for calibration, 15% for testing

X_train = X.iloc[:train_end] # train set is the first 70% of the data
y_train = y.iloc[:train_end]

X_calib = X.iloc[train_end:calib_end] # calibration set is the next 15% of the data
y_calib = y.iloc[train_end:calib_end]

X_test = X.iloc[calib_end:] # test set is the last 15% of the data
y_test = y.iloc[calib_end:]

date_train = df.iloc[:train_end]['date'] # keep track of dates for later analysis
date_calib = df.iloc[train_end:calib_end]['date'] 
date_test = df.iloc[calib_end:]['date'] 

stress_train = df.iloc[:train_end]['stress_m10'] # keep track of stress days in train set for later analysis
stress_calib = df.iloc[train_end:calib_end]['stress_m10']
stress_test = df.iloc[calib_end:]['stress_m10']

# print shapes and stress days in test set
print("Overview of train set:")
print("Rows in train:", 0, "to", train_end - 1) # 0-based indexing
print("Dates in train:", date_train.iloc[0], "to", date_train.iloc[-1]) 
print("Stress-days in train set:", stress_train.sum()) # number of stress days in train set

print("\nOverview of calibration set:")
print("Rows in calibration:", train_end, "to", calib_end - 1)
print("Dates in calibration:", date_calib.iloc[0], "to", date_calib.iloc[-1])
print("Stress-days in calibration set:", stress_calib.sum()) # number of stress days in calib set

print("\nOverview of test set:")
print("Rows in test:", calib_end, "to", n - 1)
print("Dates in test:", date_test.iloc[0], "to", date_test.iloc[-1])
print("Stress-days in test set:", stress_test.sum()) # number of stress days in test set

### Baseline CP

In [ ]:
# Fit a DT and baseline CP

from mapie.regression import SplitConformalRegressor
from sklearn.tree import DecisionTreeRegressor

dt_cp = SplitConformalRegressor(
    estimator=DecisionTreeRegressor(
        max_depth=6
        ,min_samples_leaf=10
        ,random_state=42
    )
    ,confidence_level=0.90 # alpha 0.10 (1 - alpha)
    ,conformity_score="absolute" # abs residuals to calculate conform score
    ,prefit=False # MAPIE fits the model duing cp.fit
)

dt_cp.fit(X_train, y_train) # fit

# calibrate on calibration set
dt_cp.conformalize(X_calib, y_calib)

# preds
y_pred_dt_cp, y_interval_dt_cp = dt_cp.predict_interval(X_test)

# upper and lower bounds
lower_dt_cp = y_interval_dt_cp[:, 0, 0] # lower bound
upper_dt_cp = y_interval_dt_cp[:, 1, 0] # upper bound

In [ ]:
# build results df for DT baseline CP

alpha = 0.10

results_dt_cp = pd.DataFrame({
    'date': date_test.values # build DF
    ,'actual': y_test.values # actual values
    ,'prediction': y_pred_dt_cp # point predictions
    ,'lower_bound': lower_dt_cp # lower bound
    ,'upper_bound': upper_dt_cp # upper bound
    ,'stress_m10': stress_test.values # stress flag
})

# coverage boolean value
results_dt_cp['covered'] = (
    (results_dt_cp['actual'] >= results_dt_cp['lower_bound']) & # actual is above lower bound
    (results_dt_cp['actual'] <= results_dt_cp['upper_bound']) # actual is below upper bound
)

# interwal width
results_dt_cp['interval_width'] = (
    results_dt_cp['upper_bound'] - results_dt_cp['lower_bound'] # upper bound - lower bound
)

# winkler score
results_dt_cp['winkler_score'] = np.where( # if actual value is below lower bound
    results_dt_cp['actual'] < results_dt_cp['lower_bound'],
    results_dt_cp['interval_width'] + (2 / alpha) * (results_dt_cp['lower_bound'] - results_dt_cp['actual']),
    np.where( # if actual value is above upper bound
        results_dt_cp['actual'] > results_dt_cp['upper_bound'],
        results_dt_cp['interval_width'] + (2 / alpha) * (results_dt_cp['actual'] - results_dt_cp['upper_bound']),
        results_dt_cp['interval_width'] 
    ) 
)

# 2 / alpha = 20
# if actual value is 2usd below  = 20 * 2  = 40 
# Then, 40 + interval width
#Same logic applies to upper bound
results_dt_cp.head(5)

In [ ]:
results_dt_cp[results_dt_cp['covered'] == False][
    ['actual', 'lower_bound', 'upper_bound', 'interval_width', 'winkler_score']
].head(10) # look at some examples of how winkler_score is calculated for non-covered cases

In [ ]:
# point and interval metrics

# split into stress and non-stress days to compare performance and calculate metrics separately for both groups
stress_dt_cp = results_dt_cp[results_dt_cp['stress_m10'] == 1].copy()
non_stress_dt_cp = results_dt_cp[results_dt_cp['stress_m10'] == 0].copy()

print("Stress observations", len(stress_dt_cp))
print("Non-stress observations", len(non_stress_dt_cp)) # overview

In [ ]:
# tot metrics

# point forecast metrics
mae_dt_cp_total = mean_absolute_error(results_dt_cp['actual'], results_dt_cp['prediction'])
rmse_dt_cp_total = root_mean_squared_error(results_dt_cp['actual'], results_dt_cp['prediction'])

# interval metrics
coverage_dt_cp_total = results_dt_cp['covered'].mean() # mean of all
mean_width_dt_cp_total = results_dt_cp['interval_width'].mean()
winkler_dt_cp_total = results_dt_cp['winkler_score'].mean()

# stress metrics

# point forecast metrics
mae_dt_cp_stress = mean_absolute_error(stress_dt_cp['actual'], stress_dt_cp['prediction'])
rmse_dt_cp_stress = root_mean_squared_error(stress_dt_cp['actual'], stress_dt_cp['prediction'])

# interval metrics
coverage_dt_cp_stress = stress_dt_cp['covered'].mean()
mean_width_dt_cp_stress = stress_dt_cp['interval_width'].mean()
winkler_dt_cp_stress = stress_dt_cp['winkler_score'].mean()

# non-stress metrics
# point forecast metrics
mae_dt_cp_non_stress = mean_absolute_error(non_stress_dt_cp['actual'], non_stress_dt_cp['prediction'])
rmse_dt_cp_non_stress = root_mean_squared_error(non_stress_dt_cp['actual'], non_stress_dt_cp['prediction'])
# interval metrics
coverage_dt_cp_non_stress = non_stress_dt_cp['covered'].mean()
mean_width_dt_cp_non_stress = non_stress_dt_cp['interval_width'].mean()
winkler_dt_cp_non_stress = non_stress_dt_cp['winkler_score'].mean()

 # results
print("\n*** Decision Tree + baseline CP: ***")
print("MAE:", round(mae_dt_cp_total, 4))
print("RMSE:", round(rmse_dt_cp_total, 4))
print("Coverage:", round(coverage_dt_cp_total, 4))
print("Mean Interval Width:", round(mean_width_dt_cp_total, 4))
print("Winkler Score:", round(winkler_dt_cp_total, 4))

print("\nStress period performance:")
print("MAE:", round(mae_dt_cp_stress, 4))
print("RMSE:", round(rmse_dt_cp_stress, 4))
print("Coverage:", round(coverage_dt_cp_stress, 4))
print("Mean Interval Width:", round(mean_width_dt_cp_stress, 4))
print("Winkler Score:", round(winkler_dt_cp_stress, 4))

print("\nNon-stress period performance:")
print("MAE:", round(mae_dt_cp_non_stress, 4))
print("RMSE:", round(rmse_dt_cp_non_stress, 4))
print("Coverage:", round(coverage_dt_cp_non_stress, 4))
print("Mean Interval Width:", round(mean_width_dt_cp_non_stress, 4))
print("Winkler Score:", round(winkler_dt_cp_non_stress, 4))

In [ ]:
# summary table for DT + baseline CP with variables defined above

summary_dt_cp_long = pd.DataFrame([ # long format for better plotting later on
    {
        'Model': 'Decision Tree'
        ,'Method': 'Baseline CP'
        ,'Regime': 'Total'
        ,'MAE': mae_dt_cp_total
        ,'RMSE': rmse_dt_cp_total
        ,'Coverage': coverage_dt_cp_total
        ,'Mean Interval Width': mean_width_dt_cp_total
        ,'Winkler Score': winkler_dt_cp_total
    },
    {
        'Model': 'Decision Tree'
        ,'Method': 'Baseline CP'
        ,'Regime': 'Stress'
        ,'MAE': mae_dt_cp_stress
        ,'RMSE': rmse_dt_cp_stress
        ,'Coverage': coverage_dt_cp_stress
        ,'Mean Interval Width': mean_width_dt_cp_stress
        ,'Winkler Score': winkler_dt_cp_stress
    },
    {
        'Model': 'Decision Tree'
        ,'Method': 'Baseline CP'
        ,'Regime': 'Non-Stress'
        ,'MAE': mae_dt_cp_non_stress
        ,'RMSE': rmse_dt_cp_non_stress
        ,'Coverage': coverage_dt_cp_non_stress
        ,'Mean Interval Width': mean_width_dt_cp_non_stress
        ,'Winkler Score': winkler_dt_cp_non_stress
    }
])
summary_dt_cp_long # view summary table for DT + baseline CP

#### Visualisations - Baseline

In [ ]:
# We set a color palette for the plots going forward to mimic paper-style, visually appealing plots

sns.set_theme(style="whitegrid", context="paper", font_scale=1.15)
regime_palette = {
    "Total": "#6C757D", # gray scale
    "Stress": "#A86F7A", # pinkish red
    "Non-Stress": "#ADB5BD" # lighter gray
}

edge_color = "#2F3E46" # dark gray for edge color

In [ ]:
# MAE by regime for DT + baseline CP
colors = summary_dt_cp_long["Regime"].map(regime_palette)
plt.figure(figsize=(6, 4))

plt.bar(
    summary_dt_cp_long["Regime"],
    summary_dt_cp_long["MAE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("Decision Tree + Baseline CP: MAE by Regime")
plt.xlabel("")
plt.ylabel("MAE")
plt.ylim(0, 6)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine() # remove top and right spines for cleaner look
plt.tight_layout()
plt.show()

In [ ]:
# RMSE by regime for DT + baseline CP
colors = summary_dt_cp_long["Regime"].map(regime_palette)
plt.figure(figsize=(6, 4))

plt.bar(
    summary_dt_cp_long["Regime"],
    summary_dt_cp_long["RMSE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("Decision Tree + Baseline CP: RMSE by Regime")
plt.xlabel("")
plt.ylabel("RMSE")
plt.ylim(0, 7)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Coverage by regime for DT + baseline CP

colors = summary_dt_cp_long["Regime"].map(regime_palette)
plt.figure(figsize=(6, 4))
plt.bar(
    summary_dt_cp_long["Regime"],
    summary_dt_cp_long["Coverage"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.axhline(
    y=0.90,
    color="#2F3E46",
    linestyle="--",
    linewidth=1.2,
    label="Target coverage = 0.90"
)

plt.title("Decision Tree + Baseline CP: Coverage by Regime")
plt.xlabel("")
plt.ylabel("Empirical Coverage")
plt.ylim(0, 1.0)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)
plt.legend(frameon=True)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Winkler score by regime for DT + baseline CP

colors = summary_dt_cp_long["Regime"].map(regime_palette)
plt.figure(figsize=(6, 4))

plt.bar(
    summary_dt_cp_long["Regime"],
    summary_dt_cp_long["Winkler Score"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("Decision Tree + Baseline CP: Winkler Score by Regime")
plt.xlabel("")
plt.ylabel("Winkler Score")
plt.ylim(0, 60)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine() # remove top and right spines for a cleaner visualziation
plt.tight_layout()
plt.show()

In [ ]:
# Test plot for DT + Baseline CP

plt.figure(figsize=(12, 6))

plt.fill_between(
    results_dt_cp["date"],
    results_dt_cp["lower_bound"],
    results_dt_cp["upper_bound"],
    color="#CFE3F2",
    alpha=0.35,
    label="90% Prediction Interval"
)

plt.plot(
    results_dt_cp["date"],
    results_dt_cp["actual"],
    label="Actual",
    color="#1F77B4",
    linewidth=1.5
)

plt.plot(
    results_dt_cp["date"],
    results_dt_cp["prediction"],
    label="Prediction",
    color="#FF7F0E",
    linewidth=1.4,
    alpha=0.85
)

plt.title("Decision Tree + Baseline CP: Actual vs Predicted with Prediction Intervals")
plt.xlabel("Date")
plt.ylabel("Brent Price")
plt.grid(axis='y', alpha=0.20)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# Stress period plot with zoom and markers for miss

stress_zoom_dt_cp = results_dt_cp[
    (results_dt_cp['date'] >= '2022-03-09') &
    (results_dt_cp['date'] <= '2022-05-20') # defined period
].copy()

stress_zoom_dt_cp['miss'] = (
    (stress_zoom_dt_cp['actual'] < stress_zoom_dt_cp['lower_bound']) | # miss if actual is outside the interval
    (stress_zoom_dt_cp['actual'] > stress_zoom_dt_cp['upper_bound']) # miss if actual is outside the interval
)

stress_zoom_dt_cp['hit'] = ~stress_zoom_dt_cp['miss'] # reverse above logic

misses_dt_cp = stress_zoom_dt_cp[stress_zoom_dt_cp['miss']] # subset of misses 

plt.figure(figsize=(12, 6))

plt.plot(stress_zoom_dt_cp['date'], stress_zoom_dt_cp['actual'], label='Actual', linewidth=1.5)
plt.plot(stress_zoom_dt_cp['date'], stress_zoom_dt_cp['prediction'], label='Predicted', linewidth=1.5, alpha=0.7)

plt.fill_between(
    stress_zoom_dt_cp['date']
    ,stress_zoom_dt_cp['lower_bound']
    ,stress_zoom_dt_cp['upper_bound']
    ,alpha=0.2
    ,label='Prediction interval'
)

plt.scatter(
    misses_dt_cp['date']
    ,misses_dt_cp['actual']
    ,s=50 # size of markers
    ,color='red'
    ,label='Misses'
    ,alpha=1.0 # we highlight misses more clearly
)

plt.scatter(
    stress_zoom_dt_cp[stress_zoom_dt_cp['hit']]['date']
    ,stress_zoom_dt_cp[stress_zoom_dt_cp['hit']]['actual']
    ,s=50 # size of markers
    ,color='green'
    ,label='Hits'
    ,alpha=0.4 # weaker alpha for hits, as misses are most relevant
)

plt.title('Decision Tree + Baseline Cp: Stress period zoom')
plt.xlabel('Date')
plt.ylabel('Brent Price USD')
plt.legend()
plt.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

### Decision Tree + CP EnbPI

In [ ]:
X_pretest = X.iloc[:calib_end] # all data up to the end of calibration set
y_pretest = y.iloc[:calib_end] # all data up to the end of calibration set

# EnbPI do not need a calib set in the same sense as baseline CP and we use all data up to test

In [ ]:
from mapie.regression import TimeSeriesRegressor   
from mapie.subsample import BlockBootstrap

alpha = 0.10

dt_enbpi = TimeSeriesRegressor(
    estimator=DecisionTreeRegressor( # DT with same hyperparameters as before
        max_depth=6 # limit depth
        ,min_samples_leaf=10 # atleast 10 observatiions iin each node
        ,random_state=42 # set random state for reproducibility
    )
    ,method="enbpi" 
    ,cv=BlockBootstrap(  # for handling timeseries and exchangeability violations
        n_resamplings=30 # 30 block based training sets
        ,n_blocks=10 # 10 blocks per training set
        ,overlapping=False # no overlapping between blocks for a cleaner sample
        ,random_state=42
    )
    ,agg_function="mean"  # aggregate predictions of ensemble  by taking the mean
    ,random_state=42
)
dt_enbpi.fit(X_pretest, y_pretest) # fit model


In [ ]:
# one step ahead prediction on Enbpi Update

n_test = len(X_test) # number of test observations
# the array is meant to store e.g., 100 intervalls, 2 limits upper and lower, and 1 confidence level
y_pred_dt_enbpi = np.zeros(n_test) # array to store predictions before 
y_interval_dt_enbpi = np.zeros((n_test, 2, 1)) # array to store interval bounds, 2 for bounds, and 1 for nminal level

step_size = 1 # one-step-ahead prediction

# first pred 
y_pred_dt_enbpi[:step_size], y_interval_dt_enbpi[:step_size, :, :] = dt_enbpi.predict(
    X_test.iloc[:step_size, :]
    ,confidence_level= 1 - alpha
    ,ensemble=True
)

# sequentially predict the rest of the test set, updating the model with new data after each prediction
for step in range(step_size, n_test, step_size):  # we loop through the test set in steps of 1 (one-step-ahead)
    dt_enbpi.update(
        X_test.iloc[(step - step_size): step, :]
        ,y_test.iloc[(step - step_size): step]
        ,ensemble=True # # update EnbPI with the most recent observed test point before the next one-step-ahead prediction
    )
# update enbpi with the most recent observation to adjust residual history
    y_pred_dt_enbpi[step:step + step_size], y_interval_dt_enbpi[step:step + step_size, :, :] = dt_enbpi.predict(
        X_test.iloc[step:step + step_size, :]
        ,confidence_level= 1 - alpha
        ,ensemble=True
    )

# upper and lower bounds
lower_dt_enbpi = y_interval_dt_enbpi[:, 0, 0] # lower bound
upper_dt_enbpi = y_interval_dt_enbpi[:, 1, 0] # upper bound

In [ ]:
# build results for DT + enbpi
alpha = 0.10

results_dt_enbpi = pd.DataFrame({
    'date': date_test.values
    ,'actual': y_test.values
    ,'prediction': y_pred_dt_enbpi
    ,'lower_bound': lower_dt_enbpi
    ,'upper_bound': upper_dt_enbpi
    ,'stress_m10': stress_test.values    
})

# coverage boolean value
results_dt_enbpi['covered'] = (
    (results_dt_enbpi['actual'] >= results_dt_enbpi['lower_bound']) & # actual is above lower bound
    (results_dt_enbpi['actual'] <= results_dt_enbpi['upper_bound']) # actual is below upper bound
)

# interwal width
results_dt_enbpi['interval_width'] = (
    results_dt_enbpi['upper_bound'] - results_dt_enbpi['lower_bound']
)

# winkler score
results_dt_enbpi['winkler_score'] = np.where(
    results_dt_enbpi['actual'] < results_dt_enbpi['lower_bound'],
    results_dt_enbpi['interval_width'] + (2 / alpha) * (results_dt_enbpi['lower_bound'] - results_dt_enbpi['actual']),
    np.where(
        results_dt_enbpi['actual'] > results_dt_enbpi['upper_bound'],
        results_dt_enbpi['interval_width'] + (2 / alpha) * (results_dt_enbpi['actual'] - results_dt_enbpi['upper_bound']),
        results_dt_enbpi['interval_width']
    )
)

results_dt_enbpi.head(5)

In [ ]:
# point and interval metrics
# split into stress and non-stress days to compare performance and calculate metrics separately for both groups

stress_dt_enbpi = results_dt_enbpi[results_dt_enbpi['stress_m10'] == 1].copy()
non_stress_dt_enbpi = results_dt_enbpi[results_dt_enbpi['stress_m10'] == 0].copy()
# 50 stress obs
# 904 non-stress obs

In [ ]:
# tot metrics

# point forecast metrics
mae_dt_enbpi_total = mean_absolute_error(results_dt_enbpi['actual'], results_dt_enbpi['prediction'])
rmse_dt_enbpi_total  = root_mean_squared_error(results_dt_enbpi['actual'], results_dt_enbpi['prediction'])

# interval metrics
coverage_dt_enbpi_total = results_dt_enbpi['covered'].mean()
mean_width_dt_enbpi_total = results_dt_enbpi['interval_width'].mean()
winkler_dt_enbpi_total = results_dt_enbpi['winkler_score'].mean()

# stress metrics
# point forecast metrics
mae_dt_enbpi_stress = mean_absolute_error(stress_dt_enbpi['actual'], stress_dt_enbpi['prediction'])
rmse_dt_enbpi_stress = root_mean_squared_error(stress_dt_enbpi['actual'], stress_dt_enbpi['prediction'])

# interval metrics

coverage_dt_enbpi_stress = stress_dt_enbpi['covered'].mean()
mean_width_dt_enbpi_stress = stress_dt_enbpi['interval_width'].mean()
winkler_dt_enbpi_stress = stress_dt_enbpi['winkler_score'].mean()

# non stress metrics
# point forecast metrics
mae_dt_enbpi_non_stress = mean_absolute_error(non_stress_dt_enbpi['actual'], non_stress_dt_enbpi['prediction'])
rmse_dt_enbpi_non_stress = root_mean_squared_error(non_stress_dt_enbpi['actual'], non_stress_dt_enbpi['prediction'])

# interval metrics
coverage_dt_enbpi_non_stress = non_stress_dt_enbpi['covered'].mean()
mean_width_dt_enbpi_non_stress = non_stress_dt_enbpi['interval_width'].mean()
winkler_dt_enbpi_non_stress = non_stress_dt_enbpi['winkler_score'].mean()

# results summary for DT + EnbPI
print("\n*** Decision Tree + EnbPI: ***")
print("MAE:", round(mae_dt_enbpi_total, 4))
print("RMSE:", round(rmse_dt_enbpi_total, 4))
print("Coverage:", round(coverage_dt_enbpi_total, 4))
print("Mean Interval Width:", round(mean_width_dt_enbpi_total, 4))
print("Winkler Score:", round(winkler_dt_enbpi_total, 4))

print("\nStress period performance:")
print("MAE:", round(mae_dt_enbpi_stress, 4))
print("RMSE:", round(rmse_dt_enbpi_stress, 4))
print("Coverage:", round(coverage_dt_enbpi_stress, 4))
print("Mean Interval Width:", round(mean_width_dt_enbpi_stress, 4))
print("Winkler Score:", round(winkler_dt_enbpi_stress, 4))

print("\nNon-stress period performance:")
print("MAE:", round(mae_dt_enbpi_non_stress, 4))
print("RMSE:", round(rmse_dt_enbpi_non_stress, 4))
print("Coverage:", round(coverage_dt_enbpi_non_stress, 4))
print("Mean Interval Width:", round(mean_width_dt_enbpi_non_stress, 4))
print("Winkler Score:", round(winkler_dt_enbpi_non_stress, 4))

In [ ]:
# summary table for enbpi dt

summary_dt_enbpi_long = pd.DataFrame([ # long format for better plotting later on
    {
        'Model': 'Decision Tree'
        ,'Method': 'EnbPI'
        ,'Regime': 'Total'
        ,'MAE': mae_dt_enbpi_total
        ,'RMSE': rmse_dt_enbpi_total
        ,'Coverage': coverage_dt_enbpi_total
        ,'Mean Interval Width': mean_width_dt_enbpi_total
        ,'Winkler Score': winkler_dt_enbpi_total
    },
    {
        'Model': 'Decision Tree'
        ,'Method': 'EnbPI'
        ,'Regime': 'Stress'
        ,'MAE': mae_dt_enbpi_stress
        ,'RMSE': rmse_dt_enbpi_stress
        ,'Coverage': coverage_dt_enbpi_stress
        ,'Mean Interval Width': mean_width_dt_enbpi_stress
        ,'Winkler Score': winkler_dt_enbpi_stress
    },
    {
        'Model': 'Decision Tree'
        ,'Method': 'EnbPI'
        ,'Regime': 'Non-Stress'
        ,'MAE': mae_dt_enbpi_non_stress
        ,'RMSE': rmse_dt_enbpi_non_stress
        ,'Coverage': coverage_dt_enbpi_non_stress
        ,'Mean Interval Width': mean_width_dt_enbpi_non_stress
        ,'Winkler Score': winkler_dt_enbpi_non_stress
    }
])
summary_dt_enbpi_long

#### Visualisations

In [ ]:
# MAE new plot for DT + EnbPI
colors = summary_dt_enbpi_long["Regime"].map(regime_palette)
plt.figure(figsize=(6, 4))
plt.bar(
    summary_dt_enbpi_long["Regime"],
    summary_dt_enbpi_long["MAE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)
plt.title("Decision Tree + EnbPI: MAE by Regime")
plt.xlabel("")
plt.ylabel("MAE")
plt.ylim(0, 5.5)
plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)
sns.despine() # remove top and right spines for cleaner look
plt.tight_layout()
plt.show()

In [ ]:
# RMSE plot for DT + EnbPI
colors = summary_dt_enbpi_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_dt_enbpi_long["Regime"],
    summary_dt_enbpi_long["RMSE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("Decision Tree + EnbPI: RMSE by Regime")
plt.xlabel("")
plt.ylabel("RMSE")
plt.ylim(0, 6.5)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# coveraeg

colors = summary_dt_enbpi_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_dt_enbpi_long["Regime"],
    summary_dt_enbpi_long["Coverage"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.axhline(
    y=0.90,
    color=edge_color,
    linestyle="--",
    linewidth=1.2,
    label="Target coverage = 0.90"
)

plt.title("Decision Tree + EnbPI: Coverage by Regime")
plt.xlabel("")
plt.ylabel("Empirical Coverage")
plt.ylim(0, 1.0)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

plt.legend(frameon=True)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# mean interval width for DT + EnbPI
colors = summary_dt_enbpi_long["Regime"].map(regime_palette)
plt.figure(figsize=(6, 4))
plt.bar(
    summary_dt_enbpi_long["Regime"],
    summary_dt_enbpi_long["Mean Interval Width"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("Decision Tree + EnbPI: Mean Interval Width by Regime")
plt.xlabel("")
plt.ylabel("Mean Interval Width")
plt.ylim(0, 8)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# winkler score plot for DT + EnbPI

colors = summary_dt_enbpi_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))
plt.bar(
    summary_dt_enbpi_long["Regime"],
    summary_dt_enbpi_long["Winkler Score"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)
plt.title("Decision Tree + EnbPI: Winkler Score by Regime")
plt.xlabel("")
plt.ylabel("Winkler Score")
plt.ylim(0, 55)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Test plot for DT + EnbPI

plt.figure(figsize=(12, 6))
plt.fill_between(
    results_dt_enbpi["date"],
    results_dt_enbpi["lower_bound"],
    results_dt_enbpi["upper_bound"],
    color="#CFE3F2",
    alpha=0.35,
    label="90% Prediction Interval"
)

plt.plot(
    results_dt_enbpi["date"],
    results_dt_enbpi["actual"],
    label="Actual",
    color="#1F77B4",
    linewidth=1.5
)

plt.plot(
    results_dt_enbpi["date"],
    results_dt_enbpi["prediction"],
    label="Prediction",
    color="#FF7F0E",
    linewidth=1.4,
    alpha=0.85
)

plt.title("Decision Tree + EnbPI: Actual vs Predicted with Prediction Intervals")
plt.xlabel("Date")
plt.ylabel("Brent Price")
plt.grid(axis="y", alpha=0.20)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# stress period plot for enbpi 
stress_zoom_dt_enbpi = results_dt_enbpi[
    (results_dt_enbpi['date'] >= '2022-03-09') &
    (results_dt_enbpi['date'] <= '2022-05-20')
].copy()

stress_zoom_dt_enbpi['miss'] = (
    (stress_zoom_dt_enbpi['actual'] < stress_zoom_dt_enbpi['lower_bound']) | # miss if actual is outside the interval
    (stress_zoom_dt_enbpi['actual'] > stress_zoom_dt_enbpi['upper_bound'])  # miss if actual is outside the interval
)
stress_zoom_dt_enbpi['hit'] = ~stress_zoom_dt_enbpi['miss'] # reverse logic
misses_dt_enbpi = stress_zoom_dt_enbpi[stress_zoom_dt_enbpi['miss']] 
plt.figure(figsize=(12, 6))
plt.plot(stress_zoom_dt_enbpi['date'], stress_zoom_dt_enbpi['actual'], label='Actual', linewidth=1.5)
plt.plot(stress_zoom_dt_enbpi['date'], stress_zoom_dt_enbpi['prediction'], label='Predicted', linewidth=1.5, alpha=0.7)
plt.fill_between(
    stress_zoom_dt_enbpi['date']
    ,stress_zoom_dt_enbpi['lower_bound']
    ,stress_zoom_dt_enbpi['upper_bound']
    ,alpha=0.2
    ,label='Prediction interval'
)
plt.scatter(
    misses_dt_enbpi['date']
    ,misses_dt_enbpi['actual']
    ,s=50 
    ,color='red'
    ,label='Misses'
    ,alpha=1.0
)
plt.scatter(
    stress_zoom_dt_enbpi[stress_zoom_dt_enbpi['hit']]['date']
    ,stress_zoom_dt_cp[stress_zoom_dt_enbpi['hit']]['actual']
    ,s=50 
    ,color='green'
    ,label='Hits'
    ,alpha=0.7
)
plt.title('Decision Tree + EnbPI: Stress period zoom')
plt.xlabel('Date')
plt.ylabel('Brent Price USD')
plt.legend()
plt.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

### Decision Tree + ACI 

In [ ]:
# DT + aci CP
dt_aci = TimeSeriesRegressor(
    estimator=DecisionTreeRegressor(
        max_depth=6 # same hyperparameters as before for a fair comparison
        ,min_samples_leaf=10
        ,random_state=42
)
    ,method="aci" # set method to ACI
    ,cv=BlockBootstrap(
        n_resamplings=30 # 30 block based training sets
        ,n_blocks=10  # 10 blocks per training set
        ,overlapping=False # 
        ,random_state=42
    )
    ,agg_function="mean" # aggregate predictions of ensemble by taking the mean
    ,random_state=42
)

dt_aci.fit(X_pretest, y_pretest) # fit model to the same dataset as EnbPI

In [ ]:
# one step ahead prediction for ACI

gamma = 0.005 # gamma need to be set low to not crash the entire ACI setup

n_test = len(X_test)
 # same storage as above
y_pred_dt_aci = np.zeros(n_test)
y_interval_dt_aci = np.zeros((n_test, 2, 1)) # array to store interval bounds

step_size = 1 # one-step-ahead prediction

# first pred
y_pred_dt_aci[:step_size], y_interval_dt_aci[:step_size, :, :] = dt_aci.predict(
    X_test.iloc[:step_size, :]
    ,confidence_level= 1 - alpha
    ,ensemble=True
    ,allow_infinite_bounds=False
)

# sequentially predict the rest of the test set, updating the model with new data after each prediction
for step in range(step_size, n_test, step_size):
    X_prev = X_test.iloc[(step - step_size):step, :].to_numpy() # convert to numpy array
    y_prev = y_test.iloc[(step - step_size):step].to_numpy().ravel() # ravel to make it 1D array

    dt_aci.update( # update ACI with the most recent observed test point before the next one-step-ahead prediction
        X_prev
        ,y_prev
        ,ensemble=True  
    )

    dt_aci.adapt_conformal_inference( # adapt the inference with the most recent observation to adjust residual history and gamma
        X_prev
        ,y_prev
        ,gamma=gamma
        ,confidence_level= 1 - alpha
        ,ensemble=True
    )
# make the next prediction with the updated model and inference
    y_pred_dt_aci[step:step + step_size], y_interval_dt_aci[step:step + step_size, :, :] = dt_aci.predict( 
        X_test.iloc[step:step + step_size, :],
        confidence_level=1 - alpha,
        ensemble=True,
        allow_infinite_bounds=False
    )
# upper and lower bounds
lower_dt_aci = y_interval_dt_aci[:, 0, 0] # lower bound
upper_dt_aci = y_interval_dt_aci[:, 1, 0] # upper bound

In [ ]:
#  build results for DT + ACi
alpha = 0.10

results_dt_aci = pd.DataFrame({
    'date': date_test.values
    ,'actual': y_test.values
    ,'prediction': y_pred_dt_aci
    ,'lower_bound': lower_dt_aci
    ,'upper_bound': upper_dt_aci
    ,'stress_m10': stress_test.values
})

# coverage boolean value
results_dt_aci['covered'] = (
    (results_dt_aci['actual'] >= results_dt_aci['lower_bound']) & # actual is above lower bound
    (results_dt_aci['actual'] <= results_dt_aci['upper_bound']) # actual is below upper bound
)

# interwal width
results_dt_aci['interval_width']  = (
    results_dt_aci['upper_bound'] - results_dt_aci['lower_bound']
)

# winkler
results_dt_aci['winkler_score'] = np.where(   # new column summing winkler - one score per row
    results_dt_aci['actual'] < results_dt_aci['lower_bound'], # if actual below lower, we penalize
    results_dt_aci['interval_width'] + (2 / alpha) * (results_dt_aci['lower_bound'] - results_dt_aci['actual']), # penalize
    np.where(
        results_dt_aci['actual'] > results_dt_aci['upper_bound'], # is actual above upper, we penalize
        results_dt_aci['interval_width'] + (2 / alpha) * (results_dt_aci['actual'] - results_dt_aci['upper_bound']), # penalize
        results_dt_aci['interval_width'] # if covered, winkler is just the interval width
    )
)
# 2 / alpha = 20
# if actual value is 2usd below  = 20 * 2  = 40 
# Then, 40 + interval width
#Same logic applies to upper bound
results_dt_aci.head(5)

In [ ]:
# point and interval metrics
# split into stress and non-stress days to compare performance and calculate metrics separately for both groups

stress_dt_aci = results_dt_aci[results_dt_aci['stress_m10'] == 1].copy()
non_stress_dt_aci = results_dt_aci[results_dt_aci['stress_m10'] == 0].copy()
# 50 stress obs
# 904 non-stress obs

In [ ]:
# tot metrics

# point forecast metrics
mae_dt_aci_total = mean_absolute_error(results_dt_aci['actual'], results_dt_aci['prediction'])
rmse_dt_aci_total  = root_mean_squared_error(results_dt_aci['actual'], results_dt_aci['prediction'])

# interval metrics
coverage_dt_aci_total = results_dt_aci['covered'].mean()
mean_width_dt_aci_total = results_dt_aci['interval_width'].mean()
winkler_dt_aci_total = results_dt_aci['winkler_score'].mean()

# stress metrics
# point forecast metrics
mae_dt_aci_stress = mean_absolute_error(stress_dt_aci['actual'], stress_dt_aci['prediction'])
rmse_dt_aci_stress = root_mean_squared_error(stress_dt_aci['actual'], stress_dt_aci['prediction'])

# interval metrics

coverage_dt_aci_stress = stress_dt_aci['covered'].mean()
mean_width_dt_aci_stress = stress_dt_aci['interval_width'].mean()
winkler_dt_aci_stress = stress_dt_aci['winkler_score'].mean()

# non stress metrics
# point forecast metrics
mae_dt_aci_non_stress = mean_absolute_error(non_stress_dt_aci['actual'], non_stress_dt_aci['prediction'])
rmse_dt_aci_non_stress = root_mean_squared_error(non_stress_dt_aci['actual'], non_stress_dt_aci['prediction'])

# interval metrics
coverage_dt_aci_non_stress = non_stress_dt_aci['covered'].mean()
mean_width_dt_aci_non_stress = non_stress_dt_aci['interval_width'].mean()
winkler_dt_aci_non_stress = non_stress_dt_aci['winkler_score'].mean()

# results summary for DT + EnbPI
print("\n*** Decision Tree + Aci: ***")
print("MAE:", round(mae_dt_aci_total, 4))
print("RMSE:", round(rmse_dt_aci_total, 4))
print("Coverage:", round(coverage_dt_aci_total, 4))
print("Mean Interval Width:", round(mean_width_dt_aci_total, 4))
print("Winkler Score:", round(winkler_dt_aci_total, 4))

print("\nStress period performance:")
print("MAE:", round(mae_dt_aci_stress, 4))
print("RMSE:", round(rmse_dt_aci_stress, 4))
print("Coverage:", round(coverage_dt_aci_stress, 4))
print("Mean Interval Width:", round(mean_width_dt_aci_stress, 4))
print("Winkler Score:", round(winkler_dt_aci_stress, 4))

print("\nNon-stress period performance:")
print("MAE:", round(mae_dt_aci_non_stress, 4))
print("RMSE:", round(rmse_dt_aci_non_stress, 4))
print("Coverage:", round(coverage_dt_aci_non_stress, 4))
print("Mean Interval Width:", round(mean_width_dt_aci_non_stress, 4))
print("Winkler Score:", round(winkler_dt_aci_non_stress, 4))

In [ ]:
# summary table for DT + aci

summary_dt_aci_long = pd.DataFrame([
    {
        'Model': 'Decision Tree'
        ,'Method': 'ACI'
        ,'Regime': 'Total'
        ,'MAE': mae_dt_aci_total
        ,'RMSE': rmse_dt_aci_total
        ,'Coverage': coverage_dt_aci_total
        ,'Mean Interval Width': mean_width_dt_aci_total
        ,'Winkler Score': winkler_dt_aci_total  
    },
    {
        'Model': 'Decision Tree'
        ,'Method': 'ACI'
        ,'Regime': 'Stress'
        ,'MAE': mae_dt_aci_stress
        ,'RMSE': rmse_dt_aci_stress
        ,'Coverage': coverage_dt_aci_stress
        ,'Mean Interval Width': mean_width_dt_aci_stress
        ,'Winkler Score': winkler_dt_aci_stress
    },
    {
        'Model': 'Decision Tree'
        ,'Method': 'ACI'
        ,'Regime': 'Non-Stress'
        ,'MAE': mae_dt_aci_non_stress
        ,'RMSE': rmse_dt_aci_non_stress
        ,'Coverage': coverage_dt_aci_non_stress
        ,'Mean Interval Width': mean_width_dt_aci_non_stress
        ,'Winkler Score': winkler_dt_aci_non_stress
    }
])
summary_dt_aci_long

#### Visualisations 

In [ ]:
# mae

colors = summary_dt_aci_long["Regime"].map(regime_palette)
plt.figure(figsize=(6, 4))
plt.bar(
    summary_dt_aci_long["Regime"],
    summary_dt_aci_long["MAE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)
plt.title("Decision Tree + ACI: MAE by Regime")
plt.xlabel("")
plt.ylabel("MAE")
plt.ylim(0, 5.5)
plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# rmse

colors = summary_dt_aci_long["Regime"].map(regime_palette)
plt.figure(figsize=(6, 4))
plt.bar(
    summary_dt_aci_long["Regime"],
    summary_dt_aci_long["RMSE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)
plt.title("Decision Tree + ACI: RMSE by Regime")
plt.xlabel("")
plt.ylabel("RMSE")
plt.ylim(0, 6.5)
plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# coverage plot for DT + ACI

colors = summary_dt_aci_long["Regime"].map(regime_palette)
plt.figure(figsize=(6, 4))
plt.bar(
    summary_dt_aci_long["Regime"],
    summary_dt_aci_long["Coverage"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)
plt.axhline(
    y=0.90,
    color=edge_color,
    linestyle="--",
    linewidth=1.2,
    label="Target coverage = 0.90"
)
plt.title("Decision Tree + ACI: Coverage by Regime")
plt.xlabel("")
plt.ylabel("Empirical Coverage")
plt.ylim(0, 1.0)
plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)
plt.legend(frameon=True)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# mean interval width plot for DT + ACI

colors = summary_dt_aci_long["Regime"].map(regime_palette)
plt.figure(figsize=(6, 4))
plt.bar(
    summary_dt_aci_long["Regime"],
    summary_dt_aci_long["Mean Interval Width"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)
plt.title("Decision Tree + ACI: Mean Interval Width by Regime")
plt.xlabel("")
plt.ylabel("Mean Interval Width")
plt.ylim(0, 25)
plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# winkler score for DT

colors = summary_dt_aci_long["Regime"].map(regime_palette)
plt.figure(figsize=(6, 4))
plt.bar(
    summary_dt_aci_long["Regime"],
    summary_dt_aci_long["Winkler Score"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)
plt.title("Decision Tree + ACI: Winkler Score by Regime")
plt.xlabel("")
plt.ylabel("Winkler Score")
plt.ylim(0, 40)
plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# test plot DT + aci
plt.figure(figsize=(12, 6))
plt.fill_between(
    results_dt_aci["date"],
    results_dt_aci["lower_bound"],
    results_dt_aci["upper_bound"],
    color="#CFE3F2",
    alpha=0.35,
    label="90% Prediction Interval"
)
plt.plot(
    results_dt_aci["date"],
    results_dt_aci["actual"],
    label="Actual",
    color="#1F77B4",
    linewidth=1.5
)
plt.plot(
    results_dt_aci["date"],
    results_dt_aci["prediction"],
    label="Prediction",
    color="#FF7F0E",
    linewidth=1.4,
    alpha=0.85
)
plt.title("Decision Tree + ACI: Actual vs Predicted with Prediction Intervals")
plt.xlabel("Date")
plt.ylabel("Brent Price")
plt.grid(axis="y", alpha=0.20)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# stress period plot for ACI
stress_zoom_dt_aci = results_dt_aci[
    (results_dt_aci['date'] >= '2022-03-09') &
    (results_dt_aci['date'] <= '2022-05-20')
].copy()

stress_zoom_dt_aci['miss'] = (
    (stress_zoom_dt_aci['actual'] < stress_zoom_dt_aci['lower_bound']) |  
    (stress_zoom_dt_aci['actual'] > stress_zoom_dt_aci['upper_bound']) 
)

stress_zoom_dt_aci['hit'] = ~stress_zoom_dt_aci['miss'] # flip logic
misses_dt_aci = stress_zoom_dt_aci[stress_zoom_dt_aci['miss']]

plt.figure(figsize=(12, 6))

plt.plot(
    stress_zoom_dt_aci['date'],
    stress_zoom_dt_aci['actual'],
    label='Actual',
    linewidth=1.5
)

plt.plot(
    stress_zoom_dt_aci['date'],
    stress_zoom_dt_aci['prediction'],
    label='Predicted',
    linewidth=1.5,
    alpha=0.7
)

plt.fill_between(
    stress_zoom_dt_aci['date'],
    stress_zoom_dt_aci['lower_bound'],
    stress_zoom_dt_aci['upper_bound'],
    alpha=0.2,
    label='Prediction interval'
)

plt.scatter(
    misses_dt_aci['date'],
    misses_dt_aci['actual'],
    s=50,
    color='red',
    label='Misses',
    alpha=1.0
)

plt.scatter(
    stress_zoom_dt_aci[stress_zoom_dt_aci['hit']]['date'],
    stress_zoom_dt_aci[stress_zoom_dt_aci['hit']]['actual'],
    s=50,
    color='green',
    label='Hits',
    alpha=0.7
)

plt.title('Decision Tree + ACI: Stress period zoom')
plt.xlabel('Date')
plt.ylabel('Brent Price USD')
plt.grid(axis='y', alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## Random Forest

### Baseline CP

In [ ]:
# Random Forest and baseline CP

from sklearn.ensemble import RandomForestRegressor

rf_cp = SplitConformalRegressor( # create CP object with RF as the underlying model
    estimator=RandomForestRegressor( # 
        n_estimators=300, # number of trees in the forest
        max_depth=12, # depth of each tree
        min_samples_leaf=5, # minimum samples per leaf
        random_state=42
    ),
    confidence_level=0.90,
    conformity_score="absolute",
    prefit=False
)

rf_cp.fit(X_train, y_train)

# calibration
rf_cp.conformalize(X_calib, y_calib)
# predictions
y_pred_rf_cp, y_interval_rf_cp = rf_cp.predict_interval(X_test)
# upper and lower bounds
lower_rf_cp = y_interval_rf_cp[:, 0, 0]
upper_rf_cp = y_interval_rf_cp[:, 1, 0]

In [ ]:
# results for RF + baseline CP
alpha = 0.10

results_rf_cp = pd.DataFrame({
    'date': date_test.values
    ,'actual': y_test.values
    ,'prediction': y_pred_rf_cp
    ,'lower_bound': lower_rf_cp
    ,'upper_bound': upper_rf_cp
    ,'stress_m10': stress_test.values
})
# coverage boolean value
results_rf_cp['covered'] = (
    (results_rf_cp['actual'] >= results_rf_cp['lower_bound']) & # actual is above lower bound
    (results_rf_cp['actual'] <= results_rf_cp['upper_bound']) # actual is below upper bound
)
# interwal width
results_rf_cp['interval_width'] = (
    results_rf_cp['upper_bound'] - results_rf_cp['lower_bound']
)
# winkler score
results_rf_cp['winkler_score'] = np.where(
    results_rf_cp['actual'] < results_rf_cp['lower_bound'],
    results_rf_cp['interval_width'] + (2 / alpha) * (results_rf_cp['lower_bound'] - results_rf_cp['actual']),
    np.where(
        results_rf_cp['actual'] > results_rf_cp['upper_bound'],
        results_rf_cp['interval_width'] + (2 / alpha) * (results_rf_cp['actual'] - results_rf_cp['upper_bound']),
        results_rf_cp['interval_width']
    )
)
results_rf_cp.head(2)

In [ ]:
# split stress andd non stress
stress_rf_cp = results_rf_cp[results_rf_cp['stress_m10'] == 1].copy()
non_stress_rf_cp = results_rf_cp[results_rf_cp['stress_m10'] == 0].copy()

In [ ]:
# tot metrics
# point forecast metrics
mae_rf_cp_total = mean_absolute_error(results_rf_cp['actual'], results_rf_cp['prediction'])
rmse_rf_cp_total  = root_mean_squared_error(results_rf_cp['actual'], results_rf_cp['prediction'])
# interval metrics
coverage_rf_cp_total = results_rf_cp['covered'].mean()
mean_width_rf_cp_total = results_rf_cp['interval_width'].mean()
winkler_rf_cp_total = results_rf_cp['winkler_score'].mean()
# stress metrics
# point forecast metrics
mae_rf_cp_stress = mean_absolute_error(stress_rf_cp['actual'], stress_rf_cp['prediction'])
rmse_rf_cp_stress = root_mean_squared_error(stress_rf_cp['actual'], stress_rf_cp['prediction'])
# interval metrics
coverage_rf_cp_stress = stress_rf_cp['covered'].mean()
mean_width_rf_cp_stress = stress_rf_cp['interval_width'].mean()
winkler_rf_cp_stress = stress_rf_cp['winkler_score'].mean()
# non stress metrics
# point forecast metrics
mae_rf_cp_non_stress = mean_absolute_error(non_stress_rf_cp['actual'], non_stress_rf_cp['prediction'])
rmse_rf_cp_non_stress = root_mean_squared_error(non_stress_rf_cp['actual'], non_stress_rf_cp['prediction'])
# interval metrics
coverage_rf_cp_non_stress = non_stress_rf_cp['covered'].mean()
mean_width_rf_cp_non_stress = non_stress_rf_cp['interval_width'].mean()
winkler_rf_cp_non_stress = non_stress_rf_cp['winkler_score'].mean()
# results summary for RF + Baseline CP
print("\n*** Random Forest + Baseline CP: ***")
print("MAE:", round(mae_rf_cp_total, 4))
print("RMSE:", round(rmse_rf_cp_total, 4))
print("Coverage:", round(coverage_rf_cp_total, 4))
print("Mean Interval Width:", round(mean_width_rf_cp_total, 4))
print("Winkler Score:", round(winkler_rf_cp_total, 4))
print("\nStress period performance:")
print("MAE:", round(mae_rf_cp_stress, 4))
print("RMSE:", round(rmse_rf_cp_stress, 4))
print("Coverage:", round(coverage_rf_cp_stress, 4))
print("Mean Interval Width:", round(mean_width_rf_cp_stress, 4))
print("Winkler Score:", round(winkler_rf_cp_stress, 4))
print("\nNon-stress period performance:")
print("MAE:", round(mae_rf_cp_non_stress, 4))
print("RMSE:", round(rmse_rf_cp_non_stress, 4))
print("Coverage:", round(coverage_rf_cp_non_stress, 4))
print("Mean Interval Width:", round(mean_width_rf_cp_non_stress, 4))
print("Winkler Score:", round(winkler_rf_cp_non_stress, 4))

In [ ]:
# summary table for RF + baseline CP
summary_rf_cp_long = pd.DataFrame([
    {
        'Model': 'Random Forest'
        ,'Method': 'Baseline CP'
        ,'Regime': 'Total'
        ,'MAE': mae_rf_cp_total
        ,'RMSE': rmse_rf_cp_total
        ,'Coverage': coverage_rf_cp_total
        ,'Mean Interval Width': mean_width_rf_cp_total
        ,'Winkler Score': winkler_rf_cp_total  
    },
    {
        'Model': 'Random Forest'
        ,'Method': 'Baseline CP'
        ,'Regime': 'Stress'
        ,'MAE': mae_rf_cp_stress
        ,'RMSE': rmse_rf_cp_stress
        ,'Coverage': coverage_rf_cp_stress
        ,'Mean Interval Width': mean_width_rf_cp_stress
        ,'Winkler Score': winkler_rf_cp_stress
    },
    {
        'Model': 'Random Forest'
        ,'Method': 'Baseline CP'
        ,'Regime': 'Non-Stress'
        ,'MAE': mae_rf_cp_non_stress
        ,'RMSE': rmse_rf_cp_non_stress
        ,'Coverage': coverage_rf_cp_non_stress
        ,'Mean Interval Width': mean_width_rf_cp_non_stress
        ,'Winkler Score': winkler_rf_cp_non_stress
    }
])
summary_rf_cp_long

#### Visualisations

In [ ]:
# mae

colors = summary_rf_cp_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_rf_cp_long["Regime"],
    summary_rf_cp_long["MAE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("Random Forest + Baseline CP: MAE by Regime")
plt.xlabel("")
plt.ylabel("MAE")
plt.ylim(0, 5)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# rmse

colors = summary_rf_cp_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_rf_cp_long["Regime"],
    summary_rf_cp_long["RMSE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("Random Forest + Baseline CP: RMSE by Regime")
plt.xlabel("")
plt.ylabel("RMSE")
plt.ylim(0, 6)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# coverage plot

colors = summary_rf_cp_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_rf_cp_long["Regime"],
    summary_rf_cp_long["Coverage"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.axhline(
    y=0.90,
    color=edge_color,
    linestyle="--",
    linewidth=1.2,
    label="Target coverage = 0.90"
)

plt.title("Random Forest + Baseline CP: Coverage by Regime")
plt.xlabel("")
plt.ylabel("Empirical Coverage")
plt.ylim(0, 1.0)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

plt.legend(frameon=True)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# mean interval width plot for RF + Baseline CP

colors = summary_rf_cp_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_rf_cp_long["Regime"],
    summary_rf_cp_long["Mean Interval Width"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("Random Forest + Baseline CP: Mean Interval Width by Regime")
plt.xlabel("")
plt.ylabel("Mean Interval Width")
plt.ylim(0, 7)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# winkler

colors = summary_rf_cp_long["Regime"].map(regime_palette) # use palette

plt.figure(figsize=(6, 4))

plt.bar(
    summary_rf_cp_long["Regime"],
    summary_rf_cp_long["Winkler Score"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("Random Forest + Baseline CP: Winkler Score by Regime")
plt.xlabel("")
plt.ylabel("Winkler Score")
plt.ylim(0, 55)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Test plot for RF + Baseline CP

plt.figure(figsize=(12, 6))

plt.fill_between(
    results_rf_cp["date"],
    results_rf_cp["lower_bound"],
    results_rf_cp["upper_bound"],
    color="#CFE3F2",
    alpha=0.35,
    label="90% Prediction Interval"
)

plt.plot(
    results_rf_cp["date"],
    results_rf_cp["actual"],
    label="Actual",
    color="#1F77B4",
    linewidth=1.5
)

plt.plot(
    results_rf_cp["date"],
    results_rf_cp["prediction"],
    label="Prediction",
    color="#FF7F0E",
    linewidth=1.4,
    alpha=0.85
)

plt.title("Random Forest + Baseline CP: Actual vs Predicted with Prediction Intervals")
plt.xlabel("Date")
plt.ylabel("Brent Price")
plt.grid(axis="y", alpha=0.20)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# stress period plot for RF + Baseline CP

stress_zoom_rf_cp = results_rf_cp[
    (results_rf_cp['date'] >= '2022-03-09') &
    (results_rf_cp['date'] <= '2022-05-20')
].copy()

stress_zoom_rf_cp['miss'] = (
    (stress_zoom_rf_cp['actual'] < stress_zoom_rf_cp['lower_bound']) |
    (stress_zoom_rf_cp['actual'] > stress_zoom_rf_cp['upper_bound'])
)

stress_zoom_rf_cp['hit'] = ~stress_zoom_rf_cp['miss']
misses_rf_cp = stress_zoom_rf_cp[stress_zoom_rf_cp['miss']]

plt.figure(figsize=(12, 6))

plt.plot(
    stress_zoom_rf_cp['date'],
    stress_zoom_rf_cp['actual'],
    label='Actual',
    color="#1F77B4",
    linewidth=1.5
)

plt.plot(
    stress_zoom_rf_cp['date'],
    stress_zoom_rf_cp['prediction'],
    label='Predicted',
    color="#FF7F0E",
    linewidth=1.5,
    alpha=0.7
)

plt.fill_between(
    stress_zoom_rf_cp['date'],
    stress_zoom_rf_cp['lower_bound'],
    stress_zoom_rf_cp['upper_bound'],
    color="#CFE3F2",
    alpha=0.35,
    label='Prediction interval'
)

plt.scatter(
    misses_rf_cp['date'],
    misses_rf_cp['actual'],
    s=50,
    color='red',
    label='Misses',
    alpha=1.0
)

plt.scatter(
    stress_zoom_rf_cp[stress_zoom_rf_cp['hit']]['date'],
    stress_zoom_rf_cp[stress_zoom_rf_cp['hit']]['actual'],
    s=50,
    color='green',
    label='Hits',
    alpha=0.7
)

plt.title('Random Forest + Baseline CP: Stress period zoom')
plt.xlabel('Date')
plt.ylabel('Brent Price USD')
plt.grid(axis='y', alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

### RF + Enbpi 

In [ ]:
alpha = 0.10

rf_enbpi = TimeSeriesRegressor(
    estimator=RandomForestRegressor( # RF with same hyperparameters as before
        n_estimators=300
        ,max_depth=12
        ,min_samples_leaf=5
        ,random_state=42
    )
    ,method="enbpi"
    ,cv=BlockBootstrap(  # for handling timeseries and exchangeability violations
        n_resamplings=30 # 30 block based training sets
        ,n_blocks=10 # 10 blocks per training set
        ,overlapping=False # no overlapping between blocks for a cleaner sample
        ,random_state=42
    )
    ,agg_function="mean"  # aggregate predictions of ensemble  by taking the mean
    ,random_state=42
)
rf_enbpi.fit(X_pretest, y_pretest) # fit model on same variables used above

In [ ]:
# one step ahead
# step size = 1 (already defined)
# n_test = len(X_test) (already defined)

y_pred_rf_enbpi = np.zeros(n_test)
y_interval_rf_enbpi = np.zeros((n_test, 2, 1))

#first pred
y_pred_rf_enbpi[:step_size], y_interval_rf_enbpi[:step_size, :, :] = rf_enbpi.predict(
    X_test.iloc[:step_size, :]
    ,confidence_level= 1 - alpha
    ,ensemble=True
)
#seq predict the rest of the test set, updating the model with new data after each prediction
for step in range(step_size, n_test, step_size):
    rf_enbpi.update(
        X_test.iloc[(step - step_size): step, :]
        ,y_test.iloc[(step - step_size): step]
        ,ensemble=True
    )
# update enbpi with the most recent observation to adjust residual history
    y_pred_rf_enbpi[step:step + step_size], y_interval_rf_enbpi[step:step + step_size, :, :] = rf_enbpi.predict(
        X_test.iloc[step:step + step_size, :]
        ,confidence_level= 1 - alpha
        ,ensemble=True
    )
# upper and lower bounds
lower_rf_enbpi = y_interval_rf_enbpi[:, 0, 0] # lower bound
upper_rf_enbpi = y_interval_rf_enbpi[:, 1, 0] # upper bound

In [ ]:
# results for RF + EnbPI
alpha = 0.10

results_rf_enbpi = pd.DataFrame({
    'date': date_test.values
    ,'actual': y_test.values
    ,'prediction': y_pred_rf_enbpi
    ,'lower_bound': lower_rf_enbpi
    ,'upper_bound': upper_rf_enbpi
    ,'stress_m10': stress_test.values
})
# coverage boolean value
results_rf_enbpi['covered'] = (
    (results_rf_enbpi['actual'] >= results_rf_enbpi['lower_bound']) & # actual is above lower bound
    (results_rf_enbpi['actual'] <= results_rf_enbpi['upper_bound']) # actual is below upper bound
)
# interwal width
results_rf_enbpi['interval_width'] = (
    results_rf_enbpi['upper_bound'] - results_rf_enbpi['lower_bound']
)
# winkler score
results_rf_enbpi['winkler_score'] = np.where(
    results_rf_enbpi['actual'] < results_rf_enbpi['lower_bound'],
    results_rf_enbpi['interval_width'] + (2 / alpha) * (results_rf_enbpi['lower_bound'] - results_rf_enbpi['actual']),
    np.where(
        results_rf_enbpi['actual'] > results_rf_enbpi['upper_bound'],
        results_rf_enbpi['interval_width'] + (2 / alpha) * (results_rf_enbpi['actual'] - results_rf_enbpi['upper_bound']),
        results_rf_enbpi['interval_width']
    )
)
results_rf_enbpi.head(2)

In [ ]:
# split stress and non stress
stress_rf_enbpi = results_rf_enbpi[results_rf_enbpi['stress_m10'] == 1].copy()
non_stress_rf_enbpi = results_rf_enbpi[results_rf_enbpi['stress_m10'] == 0].copy()


In [ ]:
# tot metrics
# point forecast metrics
mae_rf_enbpi_total = mean_absolute_error(results_rf_enbpi['actual'], results_rf_enbpi['prediction'])
rmse_rf_enbpi_total  = root_mean_squared_error(results_rf_enbpi['actual'], results_rf_enbpi['prediction'])
# interval metrics
coverage_rf_enbpi_total = results_rf_enbpi['covered'].mean()
mean_width_rf_enbpi_total = results_rf_enbpi['interval_width'].mean()
winkler_rf_enbpi_total = results_rf_enbpi['winkler_score'].mean()
# stress metrics
# point forecast metrics
mae_rf_enbpi_stress = mean_absolute_error(stress_rf_enbpi['actual'], stress_rf_enbpi['prediction'])
rmse_rf_enbpi_stress = root_mean_squared_error(stress_rf_enbpi['actual'], stress_rf_enbpi['prediction'])
# interval metrics
coverage_rf_enbpi_stress = stress_rf_enbpi['covered'].mean()
mean_width_rf_enbpi_stress = stress_rf_enbpi['interval_width'].mean()
winkler_rf_enbpi_stress = stress_rf_enbpi['winkler_score'].mean()
# non stress metrics
# point forecast metrics
mae_rf_enbpi_non_stress = mean_absolute_error(non_stress_rf_enbpi['actual'], non_stress_rf_enbpi['prediction'])
rmse_rf_enbpi_non_stress = root_mean_squared_error(non_stress_rf_enbpi['actual'], non_stress_rf_enbpi['prediction'])
# interval metrics
coverage_rf_enbpi_non_stress = non_stress_rf_enbpi['covered'].mean()
mean_width_rf_enbpi_non_stress = non_stress_rf_enbpi['interval_width'].mean()
winkler_rf_enbpi_non_stress = non_stress_rf_enbpi['winkler_score'].mean()
# results summary for RF + EnbPI
print("\n*** Random Forest + EnbPI: ***")
print("MAE:", round(mae_rf_enbpi_total, 4))
print("RMSE:", round(rmse_rf_enbpi_total, 4))
print("Coverage:", round(coverage_rf_enbpi_total, 4))
print("Mean Interval Width:", round(mean_width_rf_enbpi_total, 4))
print("Winkler Score:", round(winkler_rf_enbpi_total, 4))
print("\nStress period performance:")
print("MAE:", round(mae_rf_enbpi_stress, 4))
print("RMSE:", round(rmse_rf_enbpi_stress, 4))
print("Coverage:", round(coverage_rf_enbpi_stress, 4))
print("Mean Interval Width:", round(mean_width_rf_enbpi_stress, 4))
print("Winkler Score:", round(winkler_rf_enbpi_stress, 4))
print("\nNon-stress period performance:")
print("MAE:", round(mae_rf_enbpi_non_stress, 4))
print("RMSE:", round(rmse_rf_enbpi_non_stress, 4))
print("Coverage:", round(coverage_rf_enbpi_non_stress, 4))
print("Mean Interval Width:", round(mean_width_rf_enbpi_non_stress, 4))
print("Winkler Score:", round(winkler_rf_enbpi_non_stress, 4))

In [ ]:
# summary table
summary_rf_enbpi_long = pd.DataFrame([
    {
        'Model': 'Random Forest'
        ,'Method': 'EnbPI'
        ,'Regime': 'Total'
        ,'MAE': mae_rf_enbpi_total
        ,'RMSE': rmse_rf_enbpi_total
        ,'Coverage': coverage_rf_enbpi_total
        ,'Mean Interval Width': mean_width_rf_enbpi_total
        ,'Winkler Score': winkler_rf_enbpi_total  
    },
    {
        'Model': 'Random Forest'
        ,'Method': 'EnbPI'
        ,'Regime': 'Stress'
        ,'MAE': mae_rf_enbpi_stress
        ,'RMSE': rmse_rf_enbpi_stress
        ,'Coverage': coverage_rf_enbpi_stress
        ,'Mean Interval Width': mean_width_rf_enbpi_stress
        ,'Winkler Score': winkler_rf_enbpi_stress
    },
    {
        'Model': 'Random Forest'
        ,'Method': 'EnbPI'
        ,'Regime': 'Non-Stress'
        ,'MAE': mae_rf_enbpi_non_stress
        ,'RMSE': rmse_rf_enbpi_non_stress
        ,'Coverage': coverage_rf_enbpi_non_stress
        ,'Mean Interval Width': mean_width_rf_enbpi_non_stress
        ,'Winkler Score': winkler_rf_enbpi_non_stress
    }
])
summary_rf_enbpi_long

#### Visualisations 

In [ ]:
# MAE plot for RF + EnbPI

colors = summary_rf_enbpi_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_rf_enbpi_long["Regime"],
    summary_rf_enbpi_long["MAE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("Random Forest + EnbPI: MAE by Regime")
plt.xlabel("")
plt.ylabel("MAE")
plt.ylim(0, 5)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# RMSE plot for RF + EnbPI

colors = summary_rf_enbpi_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_rf_enbpi_long["Regime"],
    summary_rf_enbpi_long["RMSE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("Random Forest + EnbPI: RMSE by Regime")
plt.xlabel("")
plt.ylabel("RMSE")
plt.ylim(0, 6)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Coverage plot for RF + EnbPI

colors = summary_rf_enbpi_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_rf_enbpi_long["Regime"],
    summary_rf_enbpi_long["Coverage"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.axhline(
    y=0.90,
    color=edge_color,
    linestyle="--",
    linewidth=1.2,
    label="Target coverage = 0.90"
)

plt.title("Random Forest + EnbPI: Coverage by Regime")
plt.xlabel("")
plt.ylabel("Empirical Coverage")
plt.ylim(0, 1.0)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

plt.legend(frameon=True)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# iinterval plot for RF + EnbPI

colors = summary_rf_enbpi_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_rf_enbpi_long["Regime"],
    summary_rf_enbpi_long["Mean Interval Width"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("Random Forest + EnbPI: Mean Interval Width by Regime")
plt.xlabel("")
plt.ylabel("Mean Interval Width")
plt.ylim(0, 7)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Winkler  plot for RF + Enbpi

colors = summary_rf_enbpi_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_rf_enbpi_long["Regime"],
    summary_rf_enbpi_long["Winkler Score"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("Random Forest + EnbPI: Winkler Score by Regime")
plt.xlabel("")
plt.ylabel("Winkler Score")
plt.ylim(0, 50)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Test plot for RF + EnbPI

plt.figure(figsize=(12, 6))

plt.fill_between(
    results_rf_enbpi["date"],
    results_rf_enbpi["lower_bound"],
    results_rf_enbpi["upper_bound"],
    color="#CFE3F2",
    alpha=0.35,
    label="90% Prediction Interval"
)

plt.plot(
    results_rf_enbpi["date"],
    results_rf_enbpi["actual"],
    label="Actual",
    color="#1F77B4",
    linewidth=1.5
)

plt.plot(
    results_rf_enbpi["date"],
    results_rf_enbpi["prediction"],
    label="Prediction",
    color="#FF7F0E",
    linewidth=1.4,
    alpha=0.85
)

plt.title("Random Forest + EnbPI: Actual vs Predicted with Prediction Intervals")
plt.xlabel("Date")
plt.ylabel("Brent Price")
plt.grid(axis="y", alpha=0.20)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# stress period plot for RF + EnbPI

stress_zoom_rf_enbpi = results_rf_enbpi[
    (results_rf_enbpi['date'] >= '2022-03-09') &
    (results_rf_enbpi['date'] <= '2022-05-20')
].copy()

stress_zoom_rf_enbpi['miss'] = (
    (stress_zoom_rf_enbpi['actual'] < stress_zoom_rf_enbpi['lower_bound']) |
    (stress_zoom_rf_enbpi['actual'] > stress_zoom_rf_enbpi['upper_bound'])
)

stress_zoom_rf_enbpi['hit'] = ~stress_zoom_rf_enbpi['miss'] # flip logic to get hits
misses_rf_enbpi = stress_zoom_rf_enbpi[stress_zoom_rf_enbpi['miss']]

plt.figure(figsize=(12, 6))

plt.plot(
    stress_zoom_rf_enbpi['date'],
    stress_zoom_rf_enbpi['actual'],
    label='Actual',
    color="#1F77B4",
    linewidth=1.5
)

plt.plot(
    stress_zoom_rf_enbpi['date'],
    stress_zoom_rf_enbpi['prediction'],
    label='Predicted',
    color="#FF7F0E",
    linewidth=1.5,
    alpha=0.7
)

plt.fill_between(
    stress_zoom_rf_enbpi['date'],
    stress_zoom_rf_enbpi['lower_bound'],
    stress_zoom_rf_enbpi['upper_bound'],
    color="#CFE3F2",
    alpha=0.35,
    label='Prediction interval'
)

plt.scatter(
    misses_rf_enbpi['date'],
    misses_rf_enbpi['actual'],
    s=50,
    color='red',
    label='Misses',
    alpha=1.0
)

plt.scatter(
    stress_zoom_rf_enbpi[stress_zoom_rf_enbpi['hit']]['date'],
    stress_zoom_rf_enbpi[stress_zoom_rf_enbpi['hit']]['actual'],
    s=50,
    color='green',
    label='Hits',
    alpha=0.7
)

plt.title('Random Forest + EnbPI: Stress period zoom')
plt.xlabel('Date')
plt.ylabel('Brent Price USD')
plt.grid(axis='y', alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

### RF + ACI

In [ ]:
# Aci + RF

rf_aci = TimeSeriesRegressor(
    estimator=RandomForestRegressor(
        n_estimators=300
        ,max_depth=12 # repeat params
        ,min_samples_leaf=5
        ,random_state=42
    )
    ,method="aci" # set method to ACI
    ,cv=BlockBootstrap(
        n_resamplings=30
        ,n_blocks=10
        ,overlapping=False
        ,random_state=42
    )
    ,agg_function="mean"
    ,random_state=42
)
rf_aci.fit(X_pretest, y_pretest) # fit model to the same dataset as EnbPI

In [ ]:
# one step ahead prediction for RF + ACI
gamma = 0.005  # try same gamma

n_test = len(X_test) # we define it again

y_pred_rf_aci = np.zeros(n_test)
y_interval_rf_aci = np.zeros((n_test, 2, 1)) # array to store interval bounds
step_size = 1 # one-step-ahead prediction

# first pred
y_pred_rf_aci[:step_size], y_interval_rf_aci[:step_size, :, :] = rf_aci.predict(
    X_test.iloc[:step_size, :]
    ,confidence_level= 1 - alpha
    ,ensemble=True
    ,allow_infinite_bounds=False
)

# seq predict
for step in range(step_size, n_test, step_size):
    X_prev = X_test.iloc[(step - step_size):step, :].to_numpy()
    y_prev = y_test.iloc[(step - step_size):step].to_numpy().ravel()

    rf_aci.update(
        X_prev
        ,y_prev
        ,ensemble=True
    )
    rf_aci.adapt_conformal_inference(
        X_prev
        ,y_prev
        ,gamma=gamma
        ,confidence_level= 1 - alpha
        ,ensemble=True
    )

    y_pred_rf_aci[step:step + step_size], y_interval_rf_aci[step:step + step_size, :, :] = rf_aci.predict(
        X_test.iloc[step:step + step_size, :],
        confidence_level=1 - alpha
        ,ensemble=True
        ,allow_infinite_bounds=False
    )

# upper and lower bounds
lower_rf_aci = y_interval_rf_aci[:, 0, 0] # lower bound
upper_rf_aci = y_interval_rf_aci[:, 1, 0] # upper bound

In [ ]:
#build results for RF + ACI
alpha = 0.10
results_rf_aci = pd.DataFrame({
    'date': date_test.values
    ,'actual': y_test.values
    ,'prediction': y_pred_rf_aci
    ,'lower_bound': lower_rf_aci
    ,'upper_bound': upper_rf_aci
    ,'stress_m10': stress_test.values
})
# coverage boolean value
results_rf_aci['covered'] = (
    (results_rf_aci['actual'] >= results_rf_aci['lower_bound']) & # actual is above lower bound
    (results_rf_aci['actual'] <= results_rf_aci['upper_bound']) # actual is below upper bound
)
# interwal width
results_rf_aci['interval_width'] = (
    results_rf_aci['upper_bound'] - results_rf_aci['lower_bound']
)
# winkler score
results_rf_aci['winkler_score'] = np.where(
    results_rf_aci['actual'] < results_rf_aci['lower_bound'],
    results_rf_aci['interval_width'] + (2 / alpha) * (results_rf_aci['lower_bound'] - results_rf_aci['actual']),
    np.where(
        results_rf_aci['actual'] > results_rf_aci['upper_bound'],
        results_rf_aci['interval_width'] + (2 / alpha) * (results_rf_aci['actual'] - results_rf_aci['upper_bound']),
        results_rf_aci['interval_width']
    )
)
results_rf_aci.head(2)

In [ ]:
# non strees vs stress
stress_rf_aci = results_rf_aci[results_rf_aci['stress_m10'] == 1].copy()
non_stress_rf_aci = results_rf_aci[results_rf_aci['stress_m10'] == 0].copy()

In [ ]:
# tot metrics

mae_rf_aci_total = mean_absolute_error(results_rf_aci['actual'], results_rf_aci['prediction'])
rmse_rf_aci_total  = root_mean_squared_error(results_rf_aci['actual'], results_rf_aci['prediction'])
#interval metrics
coverage_rf_aci_total = results_rf_aci['covered'].mean()
mean_width_rf_aci_total = results_rf_aci['interval_width'].mean()
winkler_rf_aci_total = results_rf_aci['winkler_score'].mean()
#stress metrics
#point forecast metrics
mae_rf_aci_stress = mean_absolute_error(stress_rf_aci['actual'], stress_rf_aci['prediction'])
rmse_rf_aci_stress = root_mean_squared_error(stress_rf_aci['actual'], stress_rf_aci['prediction'])
#interval metrics
coverage_rf_aci_stress = stress_rf_aci['covered'].mean()
mean_width_rf_aci_stress = stress_rf_aci['interval_width'].mean()
winkler_rf_aci_stress = stress_rf_aci['winkler_score'].mean()
#non stress metrics
# point forecast metrics
mae_rf_aci_non_stress = mean_absolute_error(non_stress_rf_aci['actual'], non_stress_rf_aci['prediction'])
rmse_rf_aci_non_stress = root_mean_squared_error(non_stress_rf_aci['actual'], non_stress_rf_aci['prediction'])
# interval metrics
coverage_rf_aci_non_stress = non_stress_rf_aci['covered'].mean()
mean_width_rf_aci_non_stress = non_stress_rf_aci['interval_width'].mean()
winkler_rf_aci_non_stress = non_stress_rf_aci['winkler_score'].mean()
# results summary for RF + ACI
print("\n*** Random Forest + ACI: ***")
print("MAE:", round(mae_rf_aci_total, 4))
print("RMSE:", round(rmse_rf_aci_total, 4))
print("Coverage:", round(coverage_rf_aci_total, 4))
print("Mean Interval Width:", round(mean_width_rf_aci_total, 4))
print("Winkler Score:", round(winkler_rf_aci_total, 4))
print("\nStress period performance:")
print("MAE:", round(mae_rf_aci_stress, 4))
print("RMSE:", round(rmse_rf_aci_stress, 4))
print("Coverage:", round(coverage_rf_aci_stress, 4))
print("Mean Interval Width:", round(mean_width_rf_aci_stress, 4))
print("Winkler Score:", round(winkler_rf_aci_stress, 4))
print("\nNon-stress period performance:")
print("MAE:", round(mae_rf_aci_non_stress, 4))
print("RMSE:", round(rmse_rf_aci_non_stress, 4))
print("Coverage:", round(coverage_rf_aci_non_stress, 4))
print("Mean Interval Width:", round(mean_width_rf_aci_non_stress, 4))
print("Winkler Score:", round(winkler_rf_aci_non_stress, 4))

In [ ]:
# summary table

summary_rf_aci_long = pd.DataFrame([
    {
        'Model': 'Random Forest'
        ,'Method': 'ACI'
        ,'Regime': 'Total'
        ,'MAE': mae_rf_aci_total
        ,'RMSE': rmse_rf_aci_total
        ,'Coverage': coverage_rf_aci_total
        ,'Mean Interval Width': mean_width_rf_aci_total
        ,'Winkler Score': winkler_rf_aci_total  
    },
    {
        'Model': 'Random Forest'
        ,'Method': 'ACI'
        ,'Regime': 'Stress'
        ,'MAE': mae_rf_aci_stress
        ,'RMSE': rmse_rf_aci_stress
        ,'Coverage': coverage_rf_aci_stress
        ,'Mean Interval Width': mean_width_rf_aci_stress
        ,'Winkler Score': winkler_rf_aci_stress
    },
    {
        'Model': 'Random Forest'
        ,'Method': 'ACI'
        ,'Regime': 'Non-Stress'
        ,'MAE': mae_rf_aci_non_stress
        ,'RMSE': rmse_rf_aci_non_stress
        ,'Coverage': coverage_rf_aci_non_stress
        ,'Mean Interval Width': mean_width_rf_aci_non_stress
        ,'Winkler Score': winkler_rf_aci_non_stress
    }
])
summary_rf_aci_long


#### Visualisations 

In [ ]:
# MAE plot for RF + ACI

colors = summary_rf_aci_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_rf_aci_long["Regime"],
    summary_rf_aci_long["MAE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("Random Forest + ACI: MAE by Regime")
plt.xlabel("")
plt.ylabel("MAE")
plt.ylim(0, 5)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# RMSE plot for RF + ACI

colors = summary_rf_aci_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_rf_aci_long["Regime"],
    summary_rf_aci_long["RMSE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("Random Forest + ACI: RMSE by Regime")
plt.xlabel("")
plt.ylabel("RMSE")
plt.ylim(0, 6)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# RMSE plot for RF + ACI

colors = summary_rf_aci_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_rf_aci_long["Regime"],
    summary_rf_aci_long["RMSE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("Random Forest + ACI: RMSE by Regime")
plt.xlabel("")
plt.ylabel("RMSE")
plt.ylim(0, 6)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Coverage plot for RF + ACI

colors = summary_rf_aci_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_rf_aci_long["Regime"],
    summary_rf_aci_long["Coverage"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.axhline(
    y=0.90,
    color=edge_color,
    linestyle="--",
    linewidth=1.2,
    label="Target coverage = 0.90"
)

plt.title("Random Forest + ACI: Coverage by Regime")
plt.xlabel("")
plt.ylabel("Empirical Coverage")
plt.ylim(0, 1.0)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

plt.legend(frameon=True)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Mean Interval Width plot for RF + ACI

colors = summary_rf_aci_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_rf_aci_long["Regime"],
    summary_rf_aci_long["Mean Interval Width"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("Random Forest + ACI: Mean Interval Width by Regime")
plt.xlabel("")
plt.ylabel("Mean Interval Width")
plt.ylim(0, 16)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Winkler Score plot for RF + ACI

colors = summary_rf_aci_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_rf_aci_long["Regime"],
    summary_rf_aci_long["Winkler Score"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("Random Forest + ACI: Winkler Score by Regime")
plt.xlabel("")
plt.ylabel("Winkler Score")
plt.ylim(0, 35)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Test plot for RF + ACI

plt.figure(figsize=(12, 6))

plt.fill_between(
    results_rf_aci["date"],
    results_rf_aci["lower_bound"],
    results_rf_aci["upper_bound"],
    color="#CFE3F2",
    alpha=0.35,
    label="90% Prediction Interval"
)

plt.plot(
    results_rf_aci["date"],
    results_rf_aci["actual"],
    label="Actual",
    color="#1F77B4",
    linewidth=1.5
)

plt.plot(
    results_rf_aci["date"],
    results_rf_aci["prediction"],
    label="Prediction",
    color="#FF7F0E",
    linewidth=1.4,
    alpha=0.85
)

plt.title("Random Forest + ACI: Actual vs Predicted with Prediction Intervals")
plt.xlabel("Date")
plt.ylabel("Brent Price")
plt.grid(axis="y", alpha=0.20)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# stress period plot for RF + ACI

stress_zoom_rf_aci = results_rf_aci[
    (results_rf_aci['date'] >= '2022-03-09') &
    (results_rf_aci['date'] <= '2022-05-20')
].copy()

stress_zoom_rf_aci['miss'] = (
    (stress_zoom_rf_aci['actual'] < stress_zoom_rf_aci['lower_bound']) |
    (stress_zoom_rf_aci['actual'] > stress_zoom_rf_aci['upper_bound'])
)

stress_zoom_rf_aci['hit'] = ~stress_zoom_rf_aci['miss']
misses_rf_aci = stress_zoom_rf_aci[stress_zoom_rf_aci['miss']]

plt.figure(figsize=(12, 6))

plt.plot(
    stress_zoom_rf_aci['date'],
    stress_zoom_rf_aci['actual'],
    label='Actual',
    color="#1F77B4",
    linewidth=1.5
)

plt.plot(
    stress_zoom_rf_aci['date'],
    stress_zoom_rf_aci['prediction'],
    label='Predicted',
    color="#FF7F0E",
    linewidth=1.5,
    alpha=0.7
)

plt.fill_between(
    stress_zoom_rf_aci['date'],
    stress_zoom_rf_aci['lower_bound'],
    stress_zoom_rf_aci['upper_bound'],
    color="#CFE3F2",
    alpha=0.35,
    label='Prediction interval'
)

plt.scatter(
    misses_rf_aci['date'],
    misses_rf_aci['actual'],
    s=50,
    color='red',
    label='Misses',
    alpha=1.0
)

plt.scatter(
    stress_zoom_rf_aci[stress_zoom_rf_aci['hit']]['date'],
    stress_zoom_rf_aci[stress_zoom_rf_aci['hit']]['actual'],
    s=50,
    color='green',
    label='Hits',
    alpha=0.7
)

plt.title('Random Forest + ACI: Stress period zoom')
plt.xlabel('Date')
plt.ylabel('Brent Price USD')
plt.grid(axis='y', alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

## LightGBM

### Baseline CP

In [ ]:
# import    
from lightgbm import LGBMRegressor

lgbm_cp = SplitConformalRegressor(
    estimator=LGBMRegressor(
        n_estimators=800
        ,learning_rate=0.03
        ,max_depth=5
        ,num_leaves=30 
        ,min_child_samples=30
        ,subsample=0.9
        ,colsample_bytree=0.9
        ,random_state=42
    ),
    confidence_level=0.90
    ,conformity_score="absolute"
    ,prefit=False
)
lgbm_cp.fit(X_train, y_train)
lgbm_cp.conformalize(X_calib, y_calib)

y_pred_lgbm_cp, y_interval_lgbm_cp = lgbm_cp.predict_interval(X_test)

# upper and lower bounds
lower_lgbm_cp = y_interval_lgbm_cp[:, 0, 0] # lower bound
upper_lgbm_cp = y_interval_lgbm_cp[:, 1, 0] # upper bound

In [ ]:
alpha = 0.10

results_lgbm_cp = pd.DataFrame({
    'date': date_test.values
    ,'actual': y_test.values
    ,'prediction': y_pred_lgbm_cp
    ,'lower_bound': lower_lgbm_cp
    ,'upper_bound': upper_lgbm_cp
    ,'stress_m10': stress_test.values
})
# coverage boolean value
results_lgbm_cp['covered'] = (
    (results_lgbm_cp['actual'] >= results_lgbm_cp['lower_bound']) & # actual is above lower bound
    (results_lgbm_cp['actual'] <= results_lgbm_cp['upper_bound']) # actual is below upper bound
)
# interwal width
results_lgbm_cp['interval_width'] = (
    results_lgbm_cp['upper_bound'] - results_lgbm_cp['lower_bound']
)
# winkler score
results_lgbm_cp['winkler_score'] = np.where(
    results_lgbm_cp['actual'] < results_lgbm_cp['lower_bound'],
    results_lgbm_cp['interval_width'] + (2 / alpha) * (results_lgbm_cp['lower_bound'] - results_lgbm_cp['actual']),
    np.where(
        results_lgbm_cp['actual'] > results_lgbm_cp['upper_bound'],
        results_lgbm_cp['interval_width'] + (2 / alpha) * (results_lgbm_cp['actual'] - results_lgbm_cp['upper_bound']),
        results_lgbm_cp['interval_width']
    )
)
results_lgbm_cp.head(1)

In [ ]:
# metrics
# split stress and non stress
stress_lgbm_cp = results_lgbm_cp[results_lgbm_cp['stress_m10'] == 1].copy()
non_stress_lgbm_cp = results_lgbm_cp[results_lgbm_cp['stress_m10'] == 0].copy()

In [ ]:
# tot metrics
mae_lgbm_cp_total = mean_absolute_error(results_lgbm_cp['actual'], results_lgbm_cp['prediction'])
rmse_lgbm_cp_total  = root_mean_squared_error(results_lgbm_cp['actual'], results_lgbm_cp['prediction'])
# interval metrics
coverage_lgbm_cp_total = results_lgbm_cp['covered'].mean()
mean_width_lgbm_cp_total = results_lgbm_cp['interval_width'].mean()
winkler_lgbm_cp_total = results_lgbm_cp['winkler_score'].mean()
# stress metrics
mae_lgbm_cp_stress = mean_absolute_error(stress_lgbm_cp['actual'], stress_lgbm_cp['prediction'])
rmse_lgbm_cp_stress = root_mean_squared_error(stress_lgbm_cp['actual'], stress_lgbm_cp['prediction'])
coverage_lgbm_cp_stress = stress_lgbm_cp['covered'].mean()
mean_width_lgbm_cp_stress = stress_lgbm_cp['interval_width'].mean()
winkler_lgbm_cp_stress = stress_lgbm_cp['winkler_score'].mean()
# non stress metrics
mae_lgbm_cp_non_stress = mean_absolute_error(non_stress_lgbm_cp['actual'], non_stress_lgbm_cp['prediction'])
rmse_lgbm_cp_non_stress = root_mean_squared_error(non_stress_lgbm_cp['actual'], non_stress_lgbm_cp['prediction'])
coverage_lgbm_cp_non_stress = non_stress_lgbm_cp['covered'].mean()
mean_width_lgbm_cp_non_stress = non_stress_lgbm_cp['interval_width'].mean()
winkler_lgbm_cp_non_stress = non_stress_lgbm_cp['winkler_score'].mean()
# results summary for LGBM + CP
print("\n*** LGBM + CP: ***")
print("MAE:", round(mae_lgbm_cp_total, 4))
print("RMSE:", round(rmse_lgbm_cp_total, 4))
print("Coverage:", round(coverage_lgbm_cp_total, 4))
print("Mean Interval Width:", round(mean_width_lgbm_cp_total, 4))
print("Winkler Score:", round(winkler_lgbm_cp_total, 4))
print("\nStress period performance:")
print("MAE:", round(mae_lgbm_cp_stress, 4))
print("RMSE:", round(rmse_lgbm_cp_stress, 4))
print("Coverage:", round(coverage_lgbm_cp_stress, 4))
print("Mean Interval Width:", round(mean_width_lgbm_cp_stress, 4))
print("Winkler Score:", round(winkler_lgbm_cp_stress, 4))
print("\nNon-stress period performance:")
print("MAE:", round(mae_lgbm_cp_non_stress, 4))
print("RMSE:", round(rmse_lgbm_cp_non_stress, 4))
print("Coverage:", round(coverage_lgbm_cp_non_stress, 4))
print("Mean Interval Width:", round(mean_width_lgbm_cp_non_stress, 4))
print("Winkler Score:", round(winkler_lgbm_cp_non_stress, 4))


In [ ]:
# sum table
summary_lgbm_cp_long = pd.DataFrame([
    {
        'Model': 'LGBM'
        ,'Method': 'Baseline CP'
        ,'Regime': 'Total'
        ,'MAE': mae_lgbm_cp_total
        ,'RMSE': rmse_lgbm_cp_total
        ,'Coverage': coverage_lgbm_cp_total
        ,'Mean Interval Width': mean_width_lgbm_cp_total
        ,'Winkler Score': winkler_lgbm_cp_total  
    },
    {
        'Model': 'LGBM'
        ,'Method': 'Baseline CP'
        ,'Regime': 'Stress'
        ,'MAE': mae_lgbm_cp_stress
        ,'RMSE': rmse_lgbm_cp_stress
        ,'Coverage': coverage_lgbm_cp_stress
        ,'Mean Interval Width': mean_width_lgbm_cp_stress
        ,'Winkler Score': winkler_lgbm_cp_stress
    },
    {
        'Model': 'LGBM'
        ,'Method': 'Baseline CP'
        ,'Regime': 'Non-Stress'
        ,'MAE': mae_lgbm_cp_non_stress
        ,'RMSE': rmse_lgbm_cp_non_stress
        ,'Coverage': coverage_lgbm_cp_non_stress
        ,'Mean Interval Width': mean_width_lgbm_cp_non_stress
        ,'Winkler Score': winkler_lgbm_cp_non_stress
    }
])
summary_lgbm_cp_long

#### Visualisations - Baseline

In [ ]:
# MAE plot for LGBM + Baseline CP

colors = summary_lgbm_cp_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_lgbm_cp_long["Regime"],
    summary_lgbm_cp_long["MAE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("LGBM + Baseline CP: MAE by Regime")
plt.xlabel("")
plt.ylabel("MAE")
plt.ylim(0, 5.5)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# RMSE plot for LGBM + Baseline CP

colors = summary_lgbm_cp_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_lgbm_cp_long["Regime"],
    summary_lgbm_cp_long["RMSE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("LGBM + Baseline CP: RMSE by Regime")
plt.xlabel("")
plt.ylabel("RMSE")
plt.ylim(0, 6.5)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Coverage plot for LGBM + Baseline CP

colors = summary_lgbm_cp_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_lgbm_cp_long["Regime"],
    summary_lgbm_cp_long["Coverage"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.axhline(
    y=0.90,
    color=edge_color,
    linestyle="--",
    linewidth=1.2,
    label="Target coverage = 0.90"
)

plt.title("LGBM + Baseline CP: Coverage by Regime")
plt.xlabel("")
plt.ylabel("Empirical Coverage")
plt.ylim(0, 1.0)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

plt.legend(frameon=True)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Mean Interval Width plot for LGBM + Baseline CP

colors = summary_lgbm_cp_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_lgbm_cp_long["Regime"],
    summary_lgbm_cp_long["Mean Interval Width"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("LGBM + Baseline CP: Mean Interval Width by Regime")
plt.xlabel("")
plt.ylabel("Mean Interval Width")
plt.ylim(0, 8)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Winkler Score plot for LGBM + Baseline CP

colors = summary_lgbm_cp_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_lgbm_cp_long["Regime"],
    summary_lgbm_cp_long["Winkler Score"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("LGBM + Baseline CP: Winkler Score by Regime")
plt.xlabel("")
plt.ylabel("Winkler Score")
plt.ylim(0, 50)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Test plot for LGBM + Baseline CP

plt.figure(figsize=(12, 6))

plt.fill_between(
    results_lgbm_cp["date"],
    results_lgbm_cp["lower_bound"],
    results_lgbm_cp["upper_bound"],
    color="#CFE3F2",
    alpha=0.35,
    label="90% Prediction Interval"
)

plt.plot(
    results_lgbm_cp["date"],
    results_lgbm_cp["actual"],
    label="Actual",
    color="#1F77B4",
    linewidth=1.5
)

plt.plot(
    results_lgbm_cp["date"],
    results_lgbm_cp["prediction"],
    label="Prediction",
    color="#FF7F0E",
    linewidth=1.4,
    alpha=0.85
)

plt.title("LGBM + Baseline CP: Actual vs Predicted with Prediction Intervals")
plt.xlabel("Date")
plt.ylabel("Brent Price")
plt.grid(axis="y", alpha=0.20)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# stress period plot for LGBM + Baseline CP

stress_zoom_lgbm_cp = results_lgbm_cp[
    (results_lgbm_cp['date'] >= '2022-03-09') &
    (results_lgbm_cp['date'] <= '2022-05-20')
].copy()

stress_zoom_lgbm_cp['miss'] = (
    (stress_zoom_lgbm_cp['actual'] < stress_zoom_lgbm_cp['lower_bound']) |
    (stress_zoom_lgbm_cp['actual'] > stress_zoom_lgbm_cp['upper_bound'])
)

stress_zoom_lgbm_cp['hit'] = ~stress_zoom_lgbm_cp['miss']
misses_lgbm_cp = stress_zoom_lgbm_cp[stress_zoom_lgbm_cp['miss']]

plt.figure(figsize=(12, 6))

plt.plot(
    stress_zoom_lgbm_cp['date'],
    stress_zoom_lgbm_cp['actual'],
    label='Actual',
    color="#1F77B4",
    linewidth=1.5
)

plt.plot(
    stress_zoom_lgbm_cp['date'],
    stress_zoom_lgbm_cp['prediction'],
    label='Predicted',
    color="#FF7F0E",
    linewidth=1.5,
    alpha=0.7
)

plt.fill_between(
    stress_zoom_lgbm_cp['date'],
    stress_zoom_lgbm_cp['lower_bound'],
    stress_zoom_lgbm_cp['upper_bound'],
    color="#CFE3F2",
    alpha=0.35,
    label='Prediction interval'
)

plt.scatter(
    misses_lgbm_cp['date'],
    misses_lgbm_cp['actual'],
    s=50,
    color='red',
    label='Misses',
    alpha=1.0
)

plt.scatter(
    stress_zoom_lgbm_cp[stress_zoom_lgbm_cp['hit']]['date'],
    stress_zoom_lgbm_cp[stress_zoom_lgbm_cp['hit']]['actual'],
    s=50,
    color='green',
    label='Hits',
    alpha=0.7
)

plt.title('LGBM + Baseline CP: Stress period zoom')
plt.xlabel('Date')
plt.ylabel('Brent Price USD')
plt.grid(axis='y', alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

### LightGBM + EnbPI 

In [ ]:
# Enbpi
lgbm_enbpi = TimeSeriesRegressor(
    estimator=LGBMRegressor(
        n_estimators=800
        ,learning_rate=0.03
        ,max_depth=5
        ,num_leaves=30
        ,min_child_samples=30
        ,subsample=0.9
        ,colsample_bytree=0.9
        ,random_state=42
    )
    ,method="enbpi" # set method to EnbPI
    ,cv=BlockBootstrap(
        n_resamplings=30
        ,n_blocks=10
        ,overlapping=False
        ,random_state=42
    )
    ,agg_function="mean"
    ,random_state=42
)

In [ ]:
lgbm_enbpi.fit(X_pretest, y_pretest) # we train enbpi on all data up to the test set

In [ ]:

y_pred_lgbm_enbpi = np.zeros(n_test)
y_interval_lgbm_enbpi = np.zeros((n_test, 2, 1))

# first pred
y_pred_lgbm_enbpi[:step_size], y_interval_lgbm_enbpi[:step_size, :, :] = lgbm_enbpi.predict(
    X_test.iloc[:step_size, :]
    ,confidence_level= 1 - alpha
    ,ensemble=True
)
# seq predict as with previous models
for step in range(step_size, n_test, step_size):
    lgbm_enbpi.update(
        X_test.iloc[(step - step_size): step, :]
        ,y_test.iloc[(step - step_size): step]
        ,ensemble=True
    )

# update 
    y_pred_lgbm_enbpi[step:step + step_size], y_interval_lgbm_enbpi[step:step + step_size, :, :] = lgbm_enbpi.predict(
        X_test.iloc[step:step + step_size, :]
        ,confidence_level= 1 - alpha
        ,ensemble=True
    )    
# bounds
lower_lgbm_enbpi = y_interval_lgbm_enbpi[:, 0, 0] # lower bound
upper_lgbm_enbpi = y_interval_lgbm_enbpi[:, 1, 0] # upper bound



In [ ]:
# results for lgbm + EnbPI
results_lgbm_enbpi = pd.DataFrame({
    'date': date_test.values
    ,'actual': y_test.values
    ,'prediction': y_pred_lgbm_enbpi
    ,'lower_bound': lower_lgbm_enbpi
    ,'upper_bound': upper_lgbm_enbpi
    ,'stress_m10': stress_test.values
})
# coverage boolean value
results_lgbm_enbpi['covered'] = (
    (results_lgbm_enbpi['actual'] >= results_lgbm_enbpi['lower_bound']) & # actual is above lower bound
    (results_lgbm_enbpi['actual'] <= results_lgbm_enbpi['upper_bound']) # actual is below upper bound
)
# interwal width
results_lgbm_enbpi['interval_width'] = (
    results_lgbm_enbpi['upper_bound'] - results_lgbm_enbpi['lower_bound']
)
# winkler score
results_lgbm_enbpi['winkler_score'] = np.where(
    results_lgbm_enbpi['actual'] < results_lgbm_enbpi['lower_bound'],
    results_lgbm_enbpi['interval_width'] + (2 / alpha) * (results_lgbm_enbpi['lower_bound'] - results_lgbm_enbpi['actual']),
    np.where(
        results_lgbm_enbpi['actual'] > results_lgbm_enbpi['upper_bound'],
        results_lgbm_enbpi['interval_width'] + (2 / alpha) * (results_lgbm_enbpi['actual'] - results_lgbm_enbpi['upper_bound']),
        results_lgbm_enbpi['interval_width']
    )
)
results_lgbm_enbpi.head(1)


In [ ]:
# stress non stress
stress_lgbm_enbpi = results_lgbm_enbpi[results_lgbm_enbpi['stress_m10'] == 1].copy()
non_stress_lgbm_enbpi = results_lgbm_enbpi[results_lgbm_enbpi['stress_m10'] == 0].copy()

In [ ]:
# tot metrics
mae_lgbm_enbpi_total = mean_absolute_error(results_lgbm_enbpi['actual'], results_lgbm_enbpi['prediction'])
rmse_lgbm_enbpi_total  = root_mean_squared_error(results_lgbm_enbpi['actual'], results_lgbm_enbpi['prediction'])
# interval metrics
coverage_lgbm_enbpi_total = results_lgbm_enbpi['covered'].mean()
mean_width_lgbm_enbpi_total = results_lgbm_enbpi['interval_width'].mean()
winkler_lgbm_enbpi_total = results_lgbm_enbpi['winkler_score'].mean()
# stress metrics
mae_lgbm_enbpi_stress = mean_absolute_error(stress_lgbm_enbpi['actual'], stress_lgbm_enbpi['prediction'])
rmse_lgbm_enbpi_stress = root_mean_squared_error(stress_lgbm_enbpi['actual'], stress_lgbm_enbpi['prediction'])
coverage_lgbm_enbpi_stress = stress_lgbm_enbpi['covered'].mean()
mean_width_lgbm_enbpi_stress = stress_lgbm_enbpi['interval_width'].mean()
winkler_lgbm_enbpi_stress = stress_lgbm_enbpi['winkler_score'].mean()
# non stress metrics
mae_lgbm_enbpi_non_stress = mean_absolute_error(non_stress_lgbm_enbpi['actual'], non_stress_lgbm_enbpi['prediction'])
rmse_lgbm_enbpi_non_stress = root_mean_squared_error(non_stress_lgbm_enbpi['actual'], non_stress_lgbm_enbpi['prediction'])
coverage_lgbm_enbpi_non_stress = non_stress_lgbm_enbpi['covered'].mean()
mean_width_lgbm_enbpi_non_stress = non_stress_lgbm_enbpi['interval_width'].mean()
winkler_lgbm_enbpi_non_stress = non_stress_lgbm_enbpi['winkler_score'].mean()
# results summary for LGBM + EnbPI
print("\n*** LGBM + EnbPI: ***")
print("MAE:", round(mae_lgbm_enbpi_total, 4))
print("RMSE:", round(rmse_lgbm_enbpi_total, 4))
print("Coverage:", round(coverage_lgbm_enbpi_total, 4))
print("Mean Interval Width:", round(mean_width_lgbm_enbpi_total, 4))
print("Winkler Score:", round(winkler_lgbm_enbpi_total, 4))
print("\nStress period performance:")
print("MAE:", round(mae_lgbm_enbpi_stress, 4))
print("RMSE:", round(rmse_lgbm_enbpi_stress, 4))
print("Coverage:", round(coverage_lgbm_enbpi_stress, 4))
print("Mean Interval Width:", round(mean_width_lgbm_enbpi_stress, 4))
print("Winkler Score:", round(winkler_lgbm_enbpi_stress, 4))
print("\nNon-stress period performance:")
print("MAE:", round(mae_lgbm_enbpi_non_stress, 4))
print("RMSE:", round(rmse_lgbm_enbpi_non_stress, 4))
print("Coverage:", round(coverage_lgbm_enbpi_non_stress, 4))
print("Mean Interval Width:", round(mean_width_lgbm_enbpi_non_stress, 4))
print("Winkler Score:", round(winkler_lgbm_enbpi_non_stress, 4))


In [ ]:
# sum table
summary_lgbm_enbpi_long = pd.DataFrame([
    {
        'Model': 'LGBM'
        ,'Method': 'EnbPI'
        ,'Regime': 'Total'
        ,'MAE': mae_lgbm_enbpi_total
        ,'RMSE': rmse_lgbm_enbpi_total
        ,'Coverage': coverage_lgbm_enbpi_total
        ,'Mean Interval Width': mean_width_lgbm_enbpi_total
        ,'Winkler Score': winkler_lgbm_enbpi_total  
    },
    {
        'Model': 'LGBM'
        ,'Method': 'EnbPI'
        ,'Regime': 'Stress'
        ,'MAE': mae_lgbm_enbpi_stress
        ,'RMSE': rmse_lgbm_enbpi_stress
        ,'Coverage': coverage_lgbm_enbpi_stress
        ,'Mean Interval Width': mean_width_lgbm_enbpi_stress
        ,'Winkler Score': winkler_lgbm_enbpi_stress
    },
    {
        'Model': 'LGBM'
        ,'Method': 'EnbPI'
        ,'Regime': 'Non-Stress'
        ,'MAE': mae_lgbm_enbpi_non_stress
        ,'RMSE': rmse_lgbm_enbpi_non_stress
        ,'Coverage': coverage_lgbm_enbpi_non_stress
        ,'Mean Interval Width': mean_width_lgbm_enbpi_non_stress
        ,'Winkler Score': winkler_lgbm_enbpi_non_stress
    }
])
summary_lgbm_enbpi_long

#### Visualisations 

In [ ]:
# MAE plot for LGBM + EnbPI

colors = summary_lgbm_enbpi_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_lgbm_enbpi_long["Regime"],
    summary_lgbm_enbpi_long["MAE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("LGBM + EnbPI: MAE by Regime")
plt.xlabel("")
plt.ylabel("MAE")
plt.ylim(0, 5.8)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# RMSE plot for LGBM + EnbPI

colors = summary_lgbm_enbpi_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_lgbm_enbpi_long["Regime"],
    summary_lgbm_enbpi_long["RMSE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("LGBM + EnbPI: RMSE by Regime")
plt.xlabel("")
plt.ylabel("RMSE")
plt.ylim(0, 6.8)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Coverage plot for LGBM + EnbPI

colors = summary_lgbm_enbpi_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_lgbm_enbpi_long["Regime"],
    summary_lgbm_enbpi_long["Coverage"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.axhline(
    y=0.90,
    color=edge_color,
    linestyle="--",
    linewidth=1.2,
    label="Target coverage = 0.90"
)

plt.title("LGBM + EnbPI: Coverage by Regime")
plt.xlabel("")
plt.ylabel("Empirical Coverage")
plt.ylim(0, 1.0)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

plt.legend(frameon=True)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# mean Interval Width plot for LGBM + EnbPI

colors = summary_lgbm_enbpi_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_lgbm_enbpi_long["Regime"],
    summary_lgbm_enbpi_long["Mean Interval Width"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("LGBM + EnbPI: Mean Interval Width by Regime")
plt.xlabel("")
plt.ylabel("Mean Interval Width")
plt.ylim(0, 8.5)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Winkler Score plot for LGBM + EnbPI

colors = summary_lgbm_enbpi_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_lgbm_enbpi_long["Regime"],
    summary_lgbm_enbpi_long["Winkler Score"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("LGBM + EnbPI: Winkler Score by Regime")
plt.xlabel("")
plt.ylabel("Winkler Score")
plt.ylim(0, 60)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# teest plot for LGBM + EnbPI

plt.figure(figsize=(12, 6))

plt.fill_between(
    results_lgbm_enbpi["date"],
    results_lgbm_enbpi["lower_bound"],
    results_lgbm_enbpi["upper_bound"],
    color="#CFE3F2",
    alpha=0.35,
    label="90% Prediction Interval"
)

plt.plot(
    results_lgbm_enbpi["date"],
    results_lgbm_enbpi["actual"],
    label="Actual",
    color="#1F77B4",
    linewidth=1.5
)

plt.plot(
    results_lgbm_enbpi["date"],
    results_lgbm_enbpi["prediction"],
    label="Prediction",
    color="#FF7F0E",
    linewidth=1.4,
    alpha=0.85
)

plt.title("LGBM + EnbPI: Actual vs Predicted with Prediction Intervals")
plt.xlabel("Date")
plt.ylabel("Brent Price")
plt.grid(axis="y", alpha=0.20)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# stress period plot for LGBM + EnbPI

stress_zoom_lgbm_enbpi = results_lgbm_enbpi[
    (results_lgbm_enbpi['date'] >= '2022-03-09') &
    (results_lgbm_enbpi['date'] <= '2022-05-20')
].copy()

stress_zoom_lgbm_enbpi['miss'] = (
    (stress_zoom_lgbm_enbpi['actual'] < stress_zoom_lgbm_enbpi['lower_bound']) |
    (stress_zoom_lgbm_enbpi['actual'] > stress_zoom_lgbm_enbpi['upper_bound'])
)

stress_zoom_lgbm_enbpi['hit'] = ~stress_zoom_lgbm_enbpi['miss']
misses_lgbm_enbpi = stress_zoom_lgbm_enbpi[stress_zoom_lgbm_enbpi['miss']]

plt.figure(figsize=(12, 6))

plt.plot(
    stress_zoom_lgbm_enbpi['date'],
    stress_zoom_lgbm_enbpi['actual'],
    label='Actual',
    color="#1F77B4",
    linewidth=1.5
)

plt.plot(
    stress_zoom_lgbm_enbpi['date'],
    stress_zoom_lgbm_enbpi['prediction'],
    label='Predicted',
    color="#FF7F0E",
    linewidth=1.5,
    alpha=0.7
)

plt.fill_between(
    stress_zoom_lgbm_enbpi['date'],
    stress_zoom_lgbm_enbpi['lower_bound'],
    stress_zoom_lgbm_enbpi['upper_bound'],
    color="#CFE3F2",
    alpha=0.35,
    label='Prediction interval'
)

plt.scatter(
    misses_lgbm_enbpi['date'],
    misses_lgbm_enbpi['actual'],
    s=50,
    color='red',
    label='Misses',
    alpha=1.0
)

plt.scatter(
    stress_zoom_lgbm_enbpi[stress_zoom_lgbm_enbpi['hit']]['date'],
    stress_zoom_lgbm_enbpi[stress_zoom_lgbm_enbpi['hit']]['actual'],
    s=50,
    color='green',
    label='Hits',
    alpha=0.7
)

plt.title('LGBM + EnbPI: Stress period zoom')
plt.xlabel('Date')
plt.ylabel('Brent Price USD')
plt.grid(axis='y', alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

### LightGBM + ACI 

In [ ]:
# aci
lgbm_aci = TimeSeriesRegressor(
    estimator=LGBMRegressor(
        n_estimators=800
        ,learning_rate=0.03
        ,max_depth=5
        ,num_leaves=30
        ,min_child_samples=30
        ,subsample=0.9
        ,colsample_bytree=0.9
        ,random_state=42
    )
    ,method="aci" # set method to ACI
    ,cv=BlockBootstrap(
        n_resamplings=30
        ,n_blocks=10
        ,overlapping=False
        ,random_state=42
    )
    ,agg_function="mean"
    ,random_state=42
)
# Fit
lgbm_aci.fit(X_pretest, y_pretest) # we train aci on all data up to the test set as witth enbpi

In [ ]:
# one step
gamma = 0.005 # to avoid crash

# n_test = len(X_test) already defined
#step size = 1

y_pred_lgbm_aci = np.zeros(n_test)
y_interval_lgbm_aci = np.zeros((n_test, 2, 1))
# first pred

y_pred_lgbm_aci[:step_size], y_interval_lgbm_aci[:step_size, :, :] = lgbm_aci.predict(
    X_test.iloc[:step_size, :]
    ,confidence_level= 1 - alpha
    ,ensemble=True
    ,allow_infinite_bounds=False # to avoid infinite bounds
)

# seq preds
for step in range(step_size, n_test, step_size):
    X_prev = X_test.iloc[(step - step_size):step, :].to_numpy()
    y_prev = y_test.iloc[(step - step_size):step].to_numpy().ravel()

    lgbm_aci.update(
        X_prev
        ,y_prev
        ,ensemble=True
    )
    lgbm_aci.adapt_conformal_inference(
        X_prev
        ,y_prev
        ,gamma=gamma
        ,confidence_level= 1 - alpha
        ,ensemble=True
    )
    y_pred_lgbm_aci[step:step + step_size], y_interval_lgbm_aci[step:step + step_size, :, :] = lgbm_aci.predict(
        X_test.iloc[step:step + step_size, :]
        ,confidence_level= 1 - alpha
        ,ensemble=True
        ,allow_infinite_bounds=False # to avoid infinite bounds
)
# bounds
lower_lgbm_aci = y_interval_lgbm_aci[:, 0, 0] # lower bound
upper_lgbm_aci = y_interval_lgbm_aci[:, 1, 0] # upper bound


In [ ]:
# build results for LGBM + ACI
results_lgbm_aci = pd.DataFrame({
    'date': date_test.values
    ,'actual': y_test.values
    ,'prediction': y_pred_lgbm_aci
    ,'lower_bound': lower_lgbm_aci
    ,'upper_bound': upper_lgbm_aci
    ,'stress_m10': stress_test.values
})
# coverage boolean value
results_lgbm_aci['covered'] = (
    (results_lgbm_aci['actual'] >= results_lgbm_aci['lower_bound']) & # actual is above lower bound
    (results_lgbm_aci['actual'] <= results_lgbm_aci['upper_bound']) # actual is below upper bound
)
# interwal width
results_lgbm_aci['interval_width'] = (
    results_lgbm_aci['upper_bound'] - results_lgbm_aci['lower_bound']
)
# winkler score
results_lgbm_aci['winkler_score'] = np.where(
    results_lgbm_aci['actual'] < results_lgbm_aci['lower_bound'],
    results_lgbm_aci['interval_width'] + (2 / alpha) * (results_lgbm_aci['lower_bound'] - results_lgbm_aci['actual']),
    np.where(
        results_lgbm_aci['actual'] > results_lgbm_aci['upper_bound'],
        results_lgbm_aci['interval_width'] + (2 / alpha) * (results_lgbm_aci['actual'] - results_lgbm_aci['upper_bound']),
        results_lgbm_aci['interval_width']
    )
)
results_lgbm_aci.head(10)

In [ ]:
# stress non stress
stress_lgbm_aci = results_lgbm_aci[results_lgbm_aci['stress_m10'] == 1].copy()
non_stress_lgbm_aci = results_lgbm_aci[results_lgbm_aci['stress_m10'] == 0].copy()


In [ ]:
# tot metrics
mae_lgbm_aci_total = mean_absolute_error(results_lgbm_aci['actual'], results_lgbm_aci['prediction'])
rmse_lgbm_aci_total  = root_mean_squared_error(results_lgbm_aci['actual'], results_lgbm_aci['prediction'])
# interval metrics
coverage_lgbm_aci_total = results_lgbm_aci['covered'].mean()
mean_width_lgbm_aci_total = results_lgbm_aci['interval_width'].mean()
winkler_lgbm_aci_total = results_lgbm_aci['winkler_score'].mean()
# stress metrics
mae_lgbm_aci_stress = mean_absolute_error(stress_lgbm_aci['actual'], stress_lgbm_aci['prediction'])
rmse_lgbm_aci_stress = root_mean_squared_error(stress_lgbm_aci['actual'], stress_lgbm_aci['prediction'])
coverage_lgbm_aci_stress = stress_lgbm_aci['covered'].mean()
mean_width_lgbm_aci_stress = stress_lgbm_aci['interval_width'].mean()
winkler_lgbm_aci_stress = stress_lgbm_aci['winkler_score'].mean()
# non stress metrics
mae_lgbm_aci_non_stress = mean_absolute_error(non_stress_lgbm_aci['actual'], non_stress_lgbm_aci['prediction'])
rmse_lgbm_aci_non_stress = root_mean_squared_error(non_stress_lgbm_aci['actual'], non_stress_lgbm_aci['prediction'])
coverage_lgbm_aci_non_stress = non_stress_lgbm_aci['covered'].mean()
mean_width_lgbm_aci_non_stress = non_stress_lgbm_aci['interval_width'].mean()
winkler_lgbm_aci_non_stress = non_stress_lgbm_aci['winkler_score'].mean()
# results summary for LGBM + ACI
print("\n*** LGBM + ACI: ***")
print("MAE:", round(mae_lgbm_aci_total, 4))
print("RMSE:", round(rmse_lgbm_aci_total, 4))
print("Coverage:", round(coverage_lgbm_aci_total, 4))
print("Mean Interval Width:", round(mean_width_lgbm_aci_total, 4))
print("Winkler Score:", round(winkler_lgbm_aci_total, 4))
print("\nStress period performance:")
print("MAE:", round(mae_lgbm_aci_stress, 4))
print("RMSE:", round(rmse_lgbm_aci_stress, 4))
print("Coverage:", round(coverage_lgbm_aci_stress, 4))
print("Mean Interval Width:", round(mean_width_lgbm_aci_stress, 4))
print("Winkler Score:", round(winkler_lgbm_aci_stress, 4))
print("\nNon-stress period performance:")
print("MAE:", round(mae_lgbm_aci_non_stress, 4))
print("RMSE:", round(rmse_lgbm_aci_non_stress, 4))
print("Coverage:", round(coverage_lgbm_aci_non_stress, 4))
print("Mean Interval Width:", round(mean_width_lgbm_aci_non_stress, 4))
print("Winkler Score:", round(winkler_lgbm_aci_non_stress, 4))


In [ ]:
# summ table
summary_lgbm_aci_long = pd.DataFrame([
    {
        'Model': 'LGBM'
        ,'Method': 'ACI'
        ,'Regime': 'Total'
        ,'MAE': mae_lgbm_aci_total
        ,'RMSE': rmse_lgbm_aci_total
        ,'Coverage': coverage_lgbm_aci_total
        ,'Mean Interval Width': mean_width_lgbm_aci_total
        ,'Winkler Score': winkler_lgbm_aci_total  
    },
    {
        'Model': 'LGBM'
        ,'Method': 'ACI'
        ,'Regime': 'Stress'
        ,'MAE': mae_lgbm_aci_stress
        ,'RMSE': rmse_lgbm_aci_stress
        ,'Coverage': coverage_lgbm_aci_stress
        ,'Mean Interval Width': mean_width_lgbm_aci_stress
        ,'Winkler Score': winkler_lgbm_aci_stress
    },
    {
        'Model': 'LGBM'
        ,'Method': 'ACI'
        ,'Regime': 'Non-Stress'
        ,'MAE': mae_lgbm_aci_non_stress
        ,'RMSE': rmse_lgbm_aci_non_stress
        ,'Coverage': coverage_lgbm_aci_non_stress
        ,'Mean Interval Width': mean_width_lgbm_aci_non_stress
        ,'Winkler Score': winkler_lgbm_aci_non_stress
    }
])
summary_lgbm_aci_long

#### Visualisations 

In [ ]:
# mae
colors = summary_lgbm_aci_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_lgbm_aci_long["Regime"],
    summary_lgbm_aci_long["MAE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("LGBM + ACI: MAE by Regime")
plt.xlabel("")
plt.ylabel("MAE")
plt.ylim(0, 5.8)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# rmse
colors = summary_lgbm_aci_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_lgbm_aci_long["Regime"],
    summary_lgbm_aci_long["RMSE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("LGBM + ACI: RMSE by Regime")
plt.xlabel("")
plt.ylabel("RMSE")
plt.ylim(0, 6.8)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# #coverage plot for LGBM + ACI
colors = summary_lgbm_aci_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_lgbm_aci_long["Regime"],
    summary_lgbm_aci_long["Coverage"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.axhline(
    y=0.90,
    color=edge_color,
    linestyle="--",
    linewidth=1.2,
    label="Target coverage = 0.90"
)

plt.title("LGBM + ACI: Coverage by Regime")
plt.xlabel("")
plt.ylabel("Empirical Coverage")
plt.ylim(0, 1.0)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

plt.legend(frameon=True)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# mean width
colors = summary_lgbm_aci_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_lgbm_aci_long["Regime"],
    summary_lgbm_aci_long["Mean Interval Width"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("LGBM + ACI: Mean Interval Width by Regime")
plt.xlabel("")
plt.ylabel("Mean Interval Width")
plt.ylim(0, 16)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# winkler score plot for LGBM + ACI

colors = summary_lgbm_aci_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_lgbm_aci_long["Regime"],
    summary_lgbm_aci_long["Winkler Score"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("LGBM + ACI: Winkler Score by Regime")
plt.xlabel("")
plt.ylabel("Winkler Score")
plt.ylim(0, 25)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# test plot for LGBM + ACI

plt.figure(figsize=(12, 6))

plt.fill_between(
    results_lgbm_aci["date"],
    results_lgbm_aci["lower_bound"],
    results_lgbm_aci["upper_bound"],
    color="#CFE3F2",
    alpha=0.35,
    label="90% Prediction Interval"
)

plt.plot(
    results_lgbm_aci["date"],
    results_lgbm_aci["actual"],
    label="Actual",
    color="#1F77B4",
    linewidth=1.5
)

plt.plot(
    results_lgbm_aci["date"],
    results_lgbm_aci["prediction"],
    label="Prediction",
    color="#FF7F0E",
    linewidth=1.4,
    alpha=0.85
)

plt.title("LGBM + ACI: Actual vs Predicted with Prediction Intervals")
plt.xlabel("Date")
plt.ylabel("Brent Price")
plt.grid(axis="y", alpha=0.20)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# stress period plot for LGBM + ACI

stress_zoom_lgbm_aci = results_lgbm_aci[
    (results_lgbm_aci['date'] >= '2022-03-09') &
    (results_lgbm_aci['date'] <= '2022-05-20')
].copy()

stress_zoom_lgbm_aci['miss'] = (
    (stress_zoom_lgbm_aci['actual'] < stress_zoom_lgbm_aci['lower_bound']) |
    (stress_zoom_lgbm_aci['actual'] > stress_zoom_lgbm_aci['upper_bound'])
)

stress_zoom_lgbm_aci['hit'] = ~stress_zoom_lgbm_aci['miss']
misses_lgbm_aci = stress_zoom_lgbm_aci[stress_zoom_lgbm_aci['miss']]

plt.figure(figsize=(12, 6))

plt.plot(
    stress_zoom_lgbm_aci['date'],
    stress_zoom_lgbm_aci['actual'],
    label='Actual',
    color="#1F77B4",
    linewidth=1.5
)

plt.plot(
    stress_zoom_lgbm_aci['date'],
    stress_zoom_lgbm_aci['prediction'],
    label='Predicted',
    color="#FF7F0E",
    linewidth=1.5,
    alpha=0.7
)

plt.fill_between(
    stress_zoom_lgbm_aci['date'],
    stress_zoom_lgbm_aci['lower_bound'],
    stress_zoom_lgbm_aci['upper_bound'],
    color="#CFE3F2",
    alpha=0.35,
    label='Prediction interval'
)

plt.scatter(
    misses_lgbm_aci['date'],
    misses_lgbm_aci['actual'],
    s=50,
    color='red',
    label='Misses',
    alpha=1.0
)

plt.scatter(
    stress_zoom_lgbm_aci[stress_zoom_lgbm_aci['hit']]['date'],
    stress_zoom_lgbm_aci[stress_zoom_lgbm_aci['hit']]['actual'],
    s=50,
    color='green',
    label='Hits',
    alpha=0.7
)

plt.title('LGBM + ACI: Stress period zoom')
plt.xlabel('Date')
plt.ylabel('Brent Price USD')
plt.grid(axis='y', alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# show resuls from stress period
stress_results = results_lgbm_aci[results_lgbm_aci['stress_m10'] == 1].copy()
stress_results[['date', 'actual', 'prediction', 'lower_bound', 'upper_bound', 'covered']].head(10)

## XGBoost

### Baseline CP

In [ ]:
from xgboost import XGBRegressor

xgb_cp = SplitConformalRegressor(
    estimator=XGBRegressor(
        n_estimators=800
        ,learning_rate=0.03
        ,max_depth=5
        ,min_child_weight=3 # xgboost uses min_child_weight instead of min_child_samples, we set it to 3 to be similar to lgbm
        ,subsample=0.9
        ,colsample_bytree=0.9
        ,objective="reg:squarederror" # set objective to regression
        ,random_state=42
    )
    ,confidence_level=0.90
    ,conformity_score="absolute"
    ,prefit=False
)
xgb_cp.fit(X_train, y_train)
xgb_cp.conformalize(X_calib, y_calib)
y_pred_xgb_cp, y_interval_xgb_cp = xgb_cp.predict_interval(X_test)
# upper and lower bounds
lower_xgb_cp = y_interval_xgb_cp[:, 0, 0] # lower bound
upper_xgb_cp = y_interval_xgb_cp[:, 1, 0] # upper bound

In [ ]:
# build results for XGB + CP
results_xgb_cp = pd.DataFrame({
    'date': date_test.values
    ,'actual': y_test.values
    ,'prediction': y_pred_xgb_cp
    ,'lower_bound': lower_xgb_cp
    ,'upper_bound': upper_xgb_cp
    ,'stress_m10': stress_test.values
})
# coverage boolean value
results_xgb_cp['covered'] = (
    (results_xgb_cp['actual'] >= results_xgb_cp['lower_bound']) & # actual is above lower bound
    (results_xgb_cp['actual'] <= results_xgb_cp['upper_bound']) # actual is below upper bound
)
# interwal width
results_xgb_cp['interval_width'] = (
    results_xgb_cp['upper_bound'] - results_xgb_cp['lower_bound']
)
# winkler score
results_xgb_cp['winkler_score'] = np.where(
    results_xgb_cp['actual'] < results_xgb_cp['lower_bound'],
    results_xgb_cp['interval_width'] + (2 / alpha) * (results_xgb_cp['lower_bound'] - results_xgb_cp['actual']),
    np.where(
        results_xgb_cp['actual'] > results_xgb_cp['upper_bound'],
        results_xgb_cp['interval_width'] + (2 / alpha) * (results_xgb_cp['actual'] - results_xgb_cp['upper_bound']),
        results_xgb_cp['interval_width']
    )
)
results_xgb_cp.head(10)

In [ ]:
# stress non stress
stress_xgb_cp = results_xgb_cp[results_xgb_cp['stress_m10'] == 1].copy()
non_stress_xgb_cp = results_xgb_cp[results_xgb_cp['stress_m10'] == 0].copy()

In [ ]:
# tot metrics
mae_xgb_cp_total = mean_absolute_error(results_xgb_cp['actual'], results_xgb_cp['prediction'])
rmse_xgb_cp_total  = root_mean_squared_error(results_xgb_cp['actual'], results_xgb_cp['prediction'])
# interval metrics
coverage_xgb_cp_total = results_xgb_cp['covered'].mean()
mean_width_xgb_cp_total = results_xgb_cp['interval_width'].mean()
winkler_xgb_cp_total = results_xgb_cp['winkler_score'].mean()
# stress metrics
mae_xgb_cp_stress = mean_absolute_error(stress_xgb_cp['actual'], stress_xgb_cp['prediction'])
rmse_xgb_cp_stress = root_mean_squared_error(stress_xgb_cp['actual'], stress_xgb_cp['prediction'])
coverage_xgb_cp_stress = stress_xgb_cp['covered'].mean()
mean_width_xgb_cp_stress = stress_xgb_cp['interval_width'].mean()
winkler_xgb_cp_stress = stress_xgb_cp['winkler_score'].mean()
# non stress metrics
mae_xgb_cp_non_stress = mean_absolute_error(non_stress_xgb_cp['actual'], non_stress_xgb_cp['prediction'])
rmse_xgb_cp_non_stress = root_mean_squared_error(non_stress_xgb_cp['actual'], non_stress_xgb_cp['prediction'])
coverage_xgb_cp_non_stress = non_stress_xgb_cp['covered'].mean()
mean_width_xgb_cp_non_stress = non_stress_xgb_cp['interval_width'].mean()
winkler_xgb_cp_non_stress = non_stress_xgb_cp['winkler_score'].mean()
# results summary for XGB + CP
print("\n*** XGB + CP: ***")
print("MAE:", round(mae_xgb_cp_total, 4))
print("RMSE:", round(rmse_xgb_cp_total, 4))
print("Coverage:", round(coverage_xgb_cp_total, 4))
print("Mean Interval Width:", round(mean_width_xgb_cp_total, 4))
print("Winkler Score:", round(winkler_xgb_cp_total, 4))
print("\nStress period performance:")
print("MAE:", round(mae_xgb_cp_stress, 4))
print("RMSE:", round(rmse_xgb_cp_stress, 4))
print("Coverage:", round(coverage_xgb_cp_stress, 4))
print("Mean Interval Width:", round(mean_width_xgb_cp_stress, 4))
print("Winkler Score:", round(winkler_xgb_cp_stress, 4))
print("\nNon-stress period performance:")
print("MAE:", round(mae_xgb_cp_non_stress, 4))
print("RMSE:", round(rmse_xgb_cp_non_stress, 4))
print("Coverage:", round(coverage_xgb_cp_non_stress, 4))
print("Mean Interval Width:", round(mean_width_xgb_cp_non_stress, 4))


In [ ]:
# sum table
summary_xgb_cp_long = pd.DataFrame([
    {
        'Model': 'XGB'
        ,'Method': 'Baseline CP'
        ,'Regime': 'Total'
        ,'MAE': mae_xgb_cp_total
        ,'RMSE': rmse_xgb_cp_total
        ,'Coverage': coverage_xgb_cp_total
        ,'Mean Interval Width': mean_width_xgb_cp_total
        ,'Winkler Score': winkler_xgb_cp_total  
    },
    {
        'Model': 'XGB'
        ,'Method': 'Baseline CP'
        ,'Regime': 'Stress'
        ,'MAE': mae_xgb_cp_stress
        ,'RMSE': rmse_xgb_cp_stress
        ,'Coverage': coverage_xgb_cp_stress
        ,'Mean Interval Width': mean_width_xgb_cp_stress
        ,'Winkler Score': winkler_xgb_cp_stress
    },
    {
        'Model': 'XGB'
        ,'Method': 'Baseline CP'
        ,'Regime': 'Non-Stress'
        ,'MAE': mae_xgb_cp_non_stress
        ,'RMSE': rmse_xgb_cp_non_stress
        ,'Coverage': coverage_xgb_cp_non_stress
        ,'Mean Interval Width': mean_width_xgb_cp_non_stress
        ,'Winkler Score': winkler_xgb_cp_non_stress
    }
])
summary_xgb_cp_long

#### Visualisations 

In [ ]:
# mae

colors = summary_xgb_cp_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_xgb_cp_long["Regime"],
    summary_xgb_cp_long["MAE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("XGB + Baseline CP: MAE by Regime")
plt.xlabel("")
plt.ylabel("MAE")
plt.ylim(0, 8)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
#  rmse

colors = summary_xgb_cp_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_xgb_cp_long["Regime"],
    summary_xgb_cp_long["RMSE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("XGB + Baseline CP: RMSE by Regime")
plt.xlabel("")
plt.ylabel("RMSE")
plt.ylim(0, 9)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# coverage

colors = summary_xgb_cp_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_xgb_cp_long["Regime"],
    summary_xgb_cp_long["Coverage"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.axhline(
    y=0.90,
    color=edge_color,
    linestyle="--",
    linewidth=1.2,
    label="Target coverage = 0.90"
)

plt.title("XGB + Baseline CP: Coverage by Regime")
plt.xlabel("")
plt.ylabel("Empirical Coverage")
plt.ylim(0, 1.0)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

plt.legend(frameon=True)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# mean interval width

colors = summary_xgb_cp_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_xgb_cp_long["Regime"],
    summary_xgb_cp_long["Mean Interval Width"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("XGB + Baseline CP: Mean Interval Width by Regime")
plt.xlabel("")
plt.ylabel("Mean Interval Width")
plt.ylim(0, 10)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# winkler score plot for XGB + Baseline CP

colors = summary_xgb_cp_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_xgb_cp_long["Regime"],
    summary_xgb_cp_long["Winkler Score"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("XGB + Baseline CP: Winkler Score by Regime")
plt.xlabel("")
plt.ylabel("Winkler Score")
plt.ylim(0, 90)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# test plot for XGB + Baseline CP

plt.figure(figsize=(12, 6))

plt.fill_between(
    results_xgb_cp["date"],
    results_xgb_cp["lower_bound"],
    results_xgb_cp["upper_bound"],
    color="#CFE3F2",
    alpha=0.35,
    label="90% Prediction Interval"
)

plt.plot(
    results_xgb_cp["date"],
    results_xgb_cp["actual"],
    label="Actual",
    color="#1F77B4",
    linewidth=1.5
)

plt.plot(
    results_xgb_cp["date"],
    results_xgb_cp["prediction"],
    label="Prediction",
    color="#FF7F0E",
    linewidth=1.4,
    alpha=0.85
)

plt.title("XGB + Baseline CP: Actual vs Predicted with Prediction Intervals")
plt.xlabel("Date")
plt.ylabel("Brent Price")
plt.grid(axis="y", alpha=0.20)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# stress plot for XGB + Baseline CP

stress_zoom_xgb_cp = results_xgb_cp[
    (results_xgb_cp['date'] >= '2022-03-09') &
    (results_xgb_cp['date'] <= '2022-05-20')
].copy()

stress_zoom_xgb_cp['miss'] = (
    (stress_zoom_xgb_cp['actual'] < stress_zoom_xgb_cp['lower_bound']) |
    (stress_zoom_xgb_cp['actual'] > stress_zoom_xgb_cp['upper_bound'])
)

stress_zoom_xgb_cp['hit'] = ~stress_zoom_xgb_cp['miss']
misses_xgb_cp = stress_zoom_xgb_cp[stress_zoom_xgb_cp['miss']]

plt.figure(figsize=(12, 6))

plt.plot(
    stress_zoom_xgb_cp['date'],
    stress_zoom_xgb_cp['actual'],
    label='Actual',
    color="#1F77B4",
    linewidth=1.5
)

plt.plot(
    stress_zoom_xgb_cp['date'],
    stress_zoom_xgb_cp['prediction'],
    label='Predicted',
    color="#FF7F0E",
    linewidth=1.5,
    alpha=0.7
)

plt.fill_between(
    stress_zoom_xgb_cp['date'],
    stress_zoom_xgb_cp['lower_bound'],
    stress_zoom_xgb_cp['upper_bound'],
    color="#CFE3F2",
    alpha=0.35,
    label='Prediction interval'
)

plt.scatter(
    misses_xgb_cp['date'],
    misses_xgb_cp['actual'],
    s=50,
    color='red',
    label='Misses',
    alpha=1.0
)

plt.scatter(
    stress_zoom_xgb_cp[stress_zoom_xgb_cp['hit']]['date'],
    stress_zoom_xgb_cp[stress_zoom_xgb_cp['hit']]['actual'],
    s=50,
    color='green',
    label='Hits',
    alpha=0.7
)

plt.title('XGB + Baseline CP: Stress period zoom')
plt.xlabel('Date')
plt.ylabel('Brent Price USD')
plt.grid(axis='y', alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

### XGBoost + EnbPI 

In [ ]:
# enbpi
alpha = 0.10

xgb_enbpi = TimeSeriesRegressor(
    estimator=XGBRegressor( # XGB with same hyperparameters as before
       n_estimators=800
        ,learning_rate=0.03
        ,max_depth=5
        ,min_child_weight=3 # xgboost uses min_child_weight instead of min_child_samples, we set it to 3 to be similar to lgbm
        ,subsample=0.9
        ,colsample_bytree=0.9
        ,objective="reg:squarederror" # set objective to regression
        ,random_state=42
    )
    ,method="enbpi"
    ,cv=BlockBootstrap(  # for handling timeseries and exchangeability violations
        n_resamplings=30 # 30 block based training sets
        ,n_blocks=10 # 10 blocks per training set
        ,overlapping=False # no overlapping between blocks for a cleaner sample
        ,random_state=42
    )
    ,agg_function="mean"  # aggregate predictions of ensemble  by taking the mean
    ,random_state=42
)
# Fit
xgb_enbpi.fit(X_pretest, y_pretest) # we train enbpi on all data up to the test set to be comparable to aci

In [ ]:
# one step

y_pred_xgb_enbpi = np.zeros(n_test)
y_interval_xgb_enbpi = np.zeros((n_test, 2, 1))
# first pred
y_pred_xgb_enbpi[:step_size], y_interval_xgb_enbpi[:step_size, :, :] = xgb_enbpi.predict(
    X_test.iloc[:step_size, :]
    ,confidence_level= 1 - alpha
    ,ensemble=True
)
#seq predict
for step in range(step_size, n_test, step_size):
    xgb_enbpi.update(
        X_test.iloc[(step - step_size): step, :]
        ,y_test.iloc[(step - step_size): step]
        ,ensemble=True
    )

#update
    y_pred_xgb_enbpi[step:step + step_size], y_interval_xgb_enbpi[step:step + step_size, :, :] = xgb_enbpi.predict(
        X_test.iloc[step:step + step_size, :]
        ,confidence_level= 1 - alpha
        ,ensemble=True
    )
# upper and lower bounds
lower_xgb_enbpi = y_interval_xgb_enbpi[:, 0, 0] # lower bound
upper_xgb_enbpi = y_interval_xgb_enbpi[:, 1, 0] # upper bound

In [ ]:
# build results for XGB + EnbPI
results_xgb_enbpi = pd.DataFrame({
    'date': date_test.values
    ,'actual': y_test.values
    ,'prediction': y_pred_xgb_enbpi
    ,'lower_bound': lower_xgb_enbpi
    ,'upper_bound': upper_xgb_enbpi
    ,'stress_m10': stress_test.values
})
# coverage boolean value
results_xgb_enbpi['covered'] = (
    (results_xgb_enbpi['actual'] >= results_xgb_enbpi['lower_bound']) & # actual is above lower bound
    (results_xgb_enbpi['actual'] <= results_xgb_enbpi['upper_bound']) # actual is below upper bound
)
# interwal width
results_xgb_enbpi['interval_width'] = (
    results_xgb_enbpi['upper_bound'] - results_xgb_enbpi['lower_bound']
)
# winkler score
results_xgb_enbpi['winkler_score'] = np.where(
    results_xgb_enbpi['actual'] < results_xgb_enbpi['lower_bound'],
    results_xgb_enbpi['interval_width'] + (2 / alpha) * (results_xgb_enbpi['lower_bound'] - results_xgb_enbpi['actual']),
    np.where(
        results_xgb_enbpi['actual'] > results_xgb_enbpi['upper_bound'],
        results_xgb_enbpi['interval_width'] + (2 / alpha) * (results_xgb_enbpi['actual'] - results_xgb_enbpi['upper_bound']),
        results_xgb_enbpi['interval_width']
    )
)
results_xgb_enbpi.head(10)

In [ ]:
# stress non stress
stress_xgb_enbpi = results_xgb_enbpi[results_xgb_enbpi['stress_m10'] == 1].copy()
non_stress_xgb_enbpi = results_xgb_enbpi[results_xgb_enbpi['stress_m10'] == 0].copy()

In [ ]:
# tot metrics
mae_xgb_enbpi_total = mean_absolute_error(results_xgb_enbpi['actual'], results_xgb_enbpi['prediction'])
rmse_xgb_enbpi_total  = root_mean_squared_error(results_xgb_enbpi['actual'], results_xgb_enbpi['prediction'])
# interval metrics
coverage_xgb_enbpi_total = results_xgb_enbpi['covered'].mean()
mean_width_xgb_enbpi_total = results_xgb_enbpi['interval_width'].mean()
winkler_xgb_enbpi_total = results_xgb_enbpi['winkler_score'].mean()
# stress metrics
mae_xgb_enbpi_stress = mean_absolute_error(stress_xgb_enbpi['actual'], stress_xgb_enbpi['prediction'])
rmse_xgb_enbpi_stress = root_mean_squared_error(stress_xgb_enbpi['actual'], stress_xgb_enbpi['prediction'])
coverage_xgb_enbpi_stress = stress_xgb_enbpi['covered'].mean()
mean_width_xgb_enbpi_stress = stress_xgb_enbpi['interval_width'].mean()
winkler_xgb_enbpi_stress = stress_xgb_enbpi['winkler_score'].mean()
# non stress metrics
mae_xgb_enbpi_non_stress = mean_absolute_error(non_stress_xgb_enbpi['actual'], non_stress_xgb_enbpi['prediction'])
rmse_xgb_enbpi_non_stress = root_mean_squared_error(non_stress_xgb_enbpi['actual'], non_stress_xgb_enbpi['prediction'])
coverage_xgb_enbpi_non_stress = non_stress_xgb_enbpi['covered'].mean()
mean_width_xgb_enbpi_non_stress = non_stress_xgb_enbpi['interval_width'].mean()
winkler_xgb_enbpi_non_stress = non_stress_xgb_enbpi['winkler_score'].mean()
# results summary for XGB + EnbPI
print("\n*** XGB + EnbPI: ***")
print("MAE:", round(mae_xgb_enbpi_total, 4))
print("RMSE:", round(rmse_xgb_enbpi_total, 4))
print("Coverage:", round(coverage_xgb_enbpi_total, 4))
print("Mean Interval Width:", round(mean_width_xgb_enbpi_total, 4))
print("Winkler Score:", round(winkler_xgb_enbpi_total, 4))
print("\nStress period performance:")
print("MAE:", round(mae_xgb_enbpi_stress, 4))
print("RMSE:", round(rmse_xgb_enbpi_stress, 4))
print("Coverage:", round(coverage_xgb_enbpi_stress, 4))
print("Mean Interval Width:", round(mean_width_xgb_enbpi_stress, 4))
print("Winkler Score:", round(winkler_xgb_enbpi_stress, 4))
print("\nNon-stress period performance:")
print("MAE:", round(mae_xgb_enbpi_non_stress, 4))
print("RMSE:", round(rmse_xgb_enbpi_non_stress, 4))
print("Coverage:", round(coverage_xgb_enbpi_non_stress, 4))
print("Mean Interval Width:", round(mean_width_xgb_enbpi_non_stress, 4))
print("Winkler score:", round(winkler_xgb_enbpi_non_stress, 4))


In [ ]:
# sum table
summary_xgb_enbpi_long = pd.DataFrame([
    {
        'Model': 'XGB'
        ,'Method': 'EnbPI'
        ,'Regime': 'Total'
        ,'MAE': mae_xgb_enbpi_total
        ,'RMSE': rmse_xgb_enbpi_total
        ,'Coverage': coverage_xgb_enbpi_total
        ,'Mean Interval Width': mean_width_xgb_enbpi_total
        ,'Winkler Score': winkler_xgb_enbpi_total  
    },
    {
        'Model': 'XGB'
        ,'Method': 'EnbPI'
        ,'Regime': 'Stress'
        ,'MAE': mae_xgb_enbpi_stress
        ,'RMSE': rmse_xgb_enbpi_stress
        ,'Coverage': coverage_xgb_enbpi_stress
        ,'Mean Interval Width': mean_width_xgb_enbpi_stress
        ,'Winkler Score': winkler_xgb_enbpi_stress
    },
    {
        'Model': 'XGB'
        ,'Method': 'EnbPI'
        ,'Regime': 'Non-Stress'
        ,'MAE': mae_xgb_enbpi_non_stress
        ,'RMSE': rmse_xgb_enbpi_non_stress
        ,'Coverage': coverage_xgb_enbpi_non_stress
        ,'Mean Interval Width': mean_width_xgb_enbpi_non_stress
        ,'Winkler Score': winkler_xgb_enbpi_non_stress
    }
])
summary_xgb_enbpi_long

#### Visualisations 

In [ ]:
# MAE plot for XGB + EnbPI
colors = summary_xgb_enbpi_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_xgb_enbpi_long["Regime"],
    summary_xgb_enbpi_long["MAE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("XGB + EnbPI: MAE by Regime")
plt.xlabel("")
plt.ylabel("MAE")
plt.ylim(0, 8)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
#  rmse

colors = summary_xgb_enbpi_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_xgb_enbpi_long["Regime"],
    summary_xgb_enbpi_long["RMSE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("XGB + EnbPI: RMSE by Regime")
plt.xlabel("")
plt.ylabel("RMSE")
plt.ylim(0, 9)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# coverage

colors = summary_xgb_enbpi_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_xgb_enbpi_long["Regime"],
    summary_xgb_enbpi_long["Coverage"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.axhline(
    y=0.90,
    color=edge_color,
    linestyle="--",
    linewidth=1.2,
    label="Target coverage = 0.90"
)

plt.title("XGB + EnbPI: Coverage by Regime")
plt.xlabel("")
plt.ylabel("Empirical Coverage")
plt.ylim(0, 1.0)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

plt.legend(frameon=True)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# mean interval width plot for XGB + EnbPI

colors = summary_xgb_enbpi_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_xgb_enbpi_long["Regime"],
    summary_xgb_enbpi_long["Mean Interval Width"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("XGB + EnbPI: Mean Interval Width by Regime")
plt.xlabel("")
plt.ylabel("Mean Interval Width")
plt.ylim(0, 11.5)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# winkler

colors = summary_xgb_enbpi_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_xgb_enbpi_long["Regime"],
    summary_xgb_enbpi_long["Winkler Score"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("XGB + EnbPI: Winkler Score by Regime")
plt.xlabel("")
plt.ylabel("Winkler Score")
plt.ylim(0, 85)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# test plot for XGB + EnbPI

plt.figure(figsize=(12, 6))

plt.fill_between(
    results_xgb_enbpi["date"],
    results_xgb_enbpi["lower_bound"],
    results_xgb_enbpi["upper_bound"],
    color="#CFE3F2",
    alpha=0.35,
    label="90% Prediction Interval"
)

plt.plot(
    results_xgb_enbpi["date"],
    results_xgb_enbpi["actual"],
    label="Actual",
    color="#1F77B4",
    linewidth=1.5
)

plt.plot(
    results_xgb_enbpi["date"],
    results_xgb_enbpi["prediction"],
    label="Prediction",
    color="#FF7F0E",
    linewidth=1.4,
    alpha=0.85
)

plt.title("XGB + EnbPI: Actual vs Predicted with Prediction Intervals")
plt.xlabel("Date")
plt.ylabel("Brent Price")
plt.grid(axis="y", alpha=0.20)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# stress plot for XGB + EnbPI

stress_zoom_xgb_enbpi = results_xgb_enbpi[
    (results_xgb_enbpi['date'] >= '2022-03-09') &
    (results_xgb_enbpi['date'] <= '2022-05-20')
].copy()

stress_zoom_xgb_enbpi['miss'] = (
    (stress_zoom_xgb_enbpi['actual'] < stress_zoom_xgb_enbpi['lower_bound']) |
    (stress_zoom_xgb_enbpi['actual'] > stress_zoom_xgb_enbpi['upper_bound'])
)

stress_zoom_xgb_enbpi['hit'] = ~stress_zoom_xgb_enbpi['miss']
misses_xgb_enbpi = stress_zoom_xgb_enbpi[stress_zoom_xgb_enbpi['miss']]

plt.figure(figsize=(12, 6))

plt.plot(
    stress_zoom_xgb_enbpi['date'],
    stress_zoom_xgb_enbpi['actual'],
    label='Actual',
    color="#1F77B4",
    linewidth=1.5
)

plt.plot(
    stress_zoom_xgb_enbpi['date'],
    stress_zoom_xgb_enbpi['prediction'],
    label='Predicted',
    color="#FF7F0E",
    linewidth=1.5,
    alpha=0.7
)

plt.fill_between(
    stress_zoom_xgb_enbpi['date'],
    stress_zoom_xgb_enbpi['lower_bound'],
    stress_zoom_xgb_enbpi['upper_bound'],
    color="#CFE3F2",
    alpha=0.35,
    label='Prediction interval'
)

plt.scatter(
    misses_xgb_enbpi['date'],
    misses_xgb_enbpi['actual'],
    s=50,
    color='red',
    label='Misses',
    alpha=1.0
)

plt.scatter(
    stress_zoom_xgb_enbpi[stress_zoom_xgb_enbpi['hit']]['date'],
    stress_zoom_xgb_enbpi[stress_zoom_xgb_enbpi['hit']]['actual'],
    s=50,
    color='green',
    label='Hits',
    alpha=0.7
)

plt.title('XGB + EnbPI: Stress period zoom')
plt.xlabel('Date')
plt.ylabel('Brent Price USD')
plt.grid(axis='y', alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

### XGBoost & ACI 

In [ ]:
# aci
xgb_aci = TimeSeriesRegressor(
    estimator=XGBRegressor(
        n_estimators=800
        ,learning_rate=0.03
        ,max_depth=5
        ,min_child_weight=3 # xgboost uses min_child_weight instead of min_child_samples, we set it to 1 to be similar to lgbm
        ,subsample=0.9
        ,colsample_bytree=0.9
        ,objective="reg:squarederror" # set objective to regression
        ,random_state=42
    )
    ,method="aci" # set method to ACI
    ,cv=BlockBootstrap(
        n_resamplings=30
        ,n_blocks=10
        ,overlapping=False
        ,random_state=42
    )
    ,agg_function="mean"
    ,random_state=42
)
# Fit
xgb_aci.fit(X_pretest, y_pretest) # we train aci on all data up to the test set to be comparable to enbpi


In [ ]:
# update and adapt
gamma = 0.005 # to avoid crash

# n_test = len(X_test) already defined
# step size = 1

y_pred_xgb_aci = np.zeros(n_test)
y_interval_xgb_aci = np.zeros((n_test, 2, 1))

# first pred
y_pred_xgb_aci[:step_size, ], y_interval_xgb_aci[:step_size, :, :] = xgb_aci.predict(
    X_test.iloc[:step_size, :]
    ,confidence_level= 1 - alpha
    ,ensemble=True
    ,allow_infinite_bounds=False # to avoid infinite bounds
)

# seq preds
for step in range(step_size, n_test, step_size):
    X_prev = X_test.iloc[(step - step_size):step, :].to_numpy()
    y_prev = y_test.iloc[(step - step_size):step].to_numpy().ravel()

    xgb_aci.update(
        X_prev
        ,y_prev
        ,ensemble=True
    )
    xgb_aci.adapt_conformal_inference(
        X_prev
        ,y_prev
        ,gamma=gamma
        ,confidence_level= 1 - alpha
        ,ensemble=True
    )

    y_pred_xgb_aci[step:step + step_size], y_interval_xgb_aci[step:step + step_size, :, :] = xgb_aci.predict(
        X_test.iloc[step:step + step_size, :]
        ,confidence_level= 1 - alpha
        ,ensemble=True
        ,allow_infinite_bounds=False # to avoid infinite bounds
)

# bounds
lower_xgb_aci = y_interval_xgb_aci[:, 0, 0] # lower bound
upper_xgb_aci = y_interval_xgb_aci[:, 1, 0] # upper bound

In [ ]:
# build results for XGB + ACI
results_xgb_aci = pd.DataFrame({
    'date': date_test.values
    ,'actual': y_test.values
    ,'prediction': y_pred_xgb_aci
    ,'lower_bound': lower_xgb_aci
    ,'upper_bound': upper_xgb_aci
    ,'stress_m10': stress_test.values
})
# coverage boolean value
results_xgb_aci['covered'] = (
    (results_xgb_aci['actual'] >= results_xgb_aci['lower_bound']) & # actual is above lower bound
    (results_xgb_aci['actual'] <= results_xgb_aci['upper_bound']) # actual is below upper bound
)
# interwal width
results_xgb_aci['interval_width'] = (
    results_xgb_aci['upper_bound'] - results_xgb_aci['lower_bound']
)
# winkler score
results_xgb_aci['winkler_score'] = np.where(
    results_xgb_aci['actual'] < results_xgb_aci['lower_bound'],
    results_xgb_aci['interval_width'] + (2 / alpha) * (results_xgb_aci['lower_bound'] - results_xgb_aci['actual']),
    np.where(
        results_xgb_aci['actual'] > results_xgb_aci['upper_bound'],
        results_xgb_aci['interval_width'] + (2 / alpha) * (results_xgb_aci['actual'] - results_xgb_aci['upper_bound']),
        results_xgb_aci['interval_width']
    )
)
results_xgb_aci.head(10)

In [ ]:
# stress non stress
stress_xgb_aci = results_xgb_aci[results_xgb_aci['stress_m10'] == 1].copy()
non_stress_xgb_aci = results_xgb_aci[results_xgb_aci['stress_m10'] == 0].copy()

In [ ]:
# tot metrics
mae_xgb_aci_total = mean_absolute_error(results_xgb_aci['actual'], results_xgb_aci['prediction'])
rmse_xgb_aci_total  = root_mean_squared_error(results_xgb_aci['actual'], results_xgb_aci['prediction'])
# interval metrics
coverage_xgb_aci_total = results_xgb_aci['covered'].mean()
mean_width_xgb_aci_total = results_xgb_aci['interval_width'].mean()
winkler_xgb_aci_total = results_xgb_aci['winkler_score'].mean()
# stress metrics
mae_xgb_aci_stress = mean_absolute_error(stress_xgb_aci['actual'], stress_xgb_aci['prediction'])
rmse_xgb_aci_stress = root_mean_squared_error(stress_xgb_aci['actual'], stress_xgb_aci['prediction'])
coverage_xgb_aci_stress = stress_xgb_aci['covered'].mean()
mean_width_xgb_aci_stress = stress_xgb_aci['interval_width'].mean()
winkler_xgb_aci_stress = stress_xgb_aci['winkler_score'].mean()
# non stress metrics
mae_xgb_aci_non_stress = mean_absolute_error(non_stress_xgb_aci['actual'], non_stress_xgb_aci['prediction'])
rmse_xgb_aci_non_stress = root_mean_squared_error(non_stress_xgb_aci['actual'], non_stress_xgb_aci['prediction'])
coverage_xgb_aci_non_stress = non_stress_xgb_aci['covered'].mean()
mean_width_xgb_aci_non_stress = non_stress_xgb_aci['interval_width'].mean()
winkler_xgb_aci_non_stress = non_stress_xgb_aci['winkler_score'].mean()
# results summary for XGB + ACI
print("\n*** XGB + ACI: ***")
print("MAE:", round(mae_xgb_aci_total, 4))
print("RMSE:", round(rmse_xgb_aci_total, 4))
print("Coverage:", round(coverage_xgb_aci_total, 4))
print("Mean Interval Width:", round(mean_width_xgb_aci_total, 4))
print("Winkler Score:", round(winkler_xgb_aci_total, 4))
print("\nStress period performance:")
print("MAE:", round(mae_xgb_aci_stress, 4))
print("RMSE:", round(rmse_xgb_aci_stress, 4))
print("Coverage:", round(coverage_xgb_aci_stress, 4))
print("Mean Interval Width:", round(mean_width_xgb_aci_stress, 4))
print("Winkler Score:", round(winkler_xgb_aci_stress, 4))
print("\nNon-stress period performance:")
print("MAE:", round(mae_xgb_aci_non_stress, 4))
print("RMSE:", round(rmse_xgb_aci_non_stress, 4))
print("Coverage:", round(coverage_xgb_aci_non_stress, 4))
print("Mean Interval Width:", round(mean_width_xgb_aci_non_stress, 4))
print("Winkler Score:", round(winkler_xgb_aci_non_stress, 4))


In [ ]:
# sum table
summary_xgb_aci_long = pd.DataFrame([
    {
        'Model': 'XGB'
        ,'Method': 'ACI'
        ,'Regime': 'Total'
        ,'MAE': mae_xgb_aci_total
        ,'RMSE': rmse_xgb_aci_total
        ,'Coverage': coverage_xgb_aci_total
        ,'Mean Interval Width': mean_width_xgb_aci_total
        ,'Winkler Score': winkler_xgb_aci_total  
    },
    {
        'Model': 'XGB'
        ,'Method': 'ACI'
        ,'Regime': 'Stress'
        ,'MAE': mae_xgb_aci_stress
        ,'RMSE': rmse_xgb_aci_stress
        ,'Coverage': coverage_xgb_aci_stress
        ,'Mean Interval Width': mean_width_xgb_aci_stress
        ,'Winkler Score': winkler_xgb_aci_stress
    },
    {
        'Model': 'XGB'
        ,'Method': 'ACI'
        ,'Regime': 'Non-Stress'
        ,'MAE': mae_xgb_aci_non_stress
        ,'RMSE': rmse_xgb_aci_non_stress
        ,'Coverage': coverage_xgb_aci_non_stress
        ,'Mean Interval Width': mean_width_xgb_aci_non_stress
        ,'Winkler Score': winkler_xgb_aci_non_stress
    }
])
summary_xgb_aci_long

#### Visualisations 


In [ ]:
# mae

colors = summary_xgb_aci_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_xgb_aci_long["Regime"],
    summary_xgb_aci_long["MAE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("XGB + ACI: MAE by Regime")
plt.xlabel("")
plt.ylabel("MAE")
plt.ylim(0, 8)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# rmse

colors = summary_xgb_aci_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_xgb_aci_long["Regime"],
    summary_xgb_aci_long["RMSE"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("XGB + ACI: RMSE by Regime")
plt.xlabel("")
plt.ylabel("RMSE")
plt.ylim(0, 9)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

# coverage plot for XGB + ACI
colors = summary_xgb_aci_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_xgb_aci_long["Regime"],
    summary_xgb_aci_long["Coverage"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.axhline(
    y=0.90,
    color=edge_color,
    linestyle="--",
    linewidth=1.2,
    label="Target coverage = 0.90"
)

plt.title("XGB + ACI: Coverage by Regime")
plt.xlabel("")
plt.ylabel("Empirical Coverage")
plt.ylim(0, 1.0)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

plt.legend(frameon=True)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# mean width plot for XGB + ACI
colors = summary_xgb_aci_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_xgb_aci_long["Regime"],
    summary_xgb_aci_long["Mean Interval Width"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("XGB + ACI: Mean Interval Width by Regime")
plt.xlabel("")
plt.ylabel("Mean Interval Width")
plt.ylim(0, 18)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# winklerscore plot for XGB + ACI

colors = summary_xgb_aci_long["Regime"].map(regime_palette)

plt.figure(figsize=(6, 4))

plt.bar(
    summary_xgb_aci_long["Regime"],
    summary_xgb_aci_long["Winkler Score"],
    color=colors,
    edgecolor=edge_color,
    linewidth=0.8
)

plt.title("XGB + ACI: Winkler Score by Regime")
plt.xlabel("")
plt.ylabel("Winkler Score")
plt.ylim(0, 35)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", visible=False)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# test plot for XGB + ACI

plt.figure(figsize=(12, 6))

plt.fill_between(
    results_xgb_aci["date"],
    results_xgb_aci["lower_bound"],
    results_xgb_aci["upper_bound"],
    color="#CFE3F2",
    alpha=0.35,
    label="90% Prediction Interval"
)

plt.plot(
    results_xgb_aci["date"],
    results_xgb_aci["actual"],
    label="Actual",
    color="#1F77B4",
    linewidth=1.5
)

plt.plot(
    results_xgb_aci["date"],
    results_xgb_aci["prediction"],
    label="Prediction",
    color="#FF7F0E",
    linewidth=1.4,
    alpha=0.85
)

plt.title("XGB + ACI: Actual vs Predicted with Prediction Intervals")
plt.xlabel("Date")
plt.ylabel("Brent Price")
plt.grid(axis="y", alpha=0.20)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

In [ ]:
# stressperiod plot for XGB + ACI

stress_zoom_xgb_aci = results_xgb_aci[
    (results_xgb_aci['date'] >= '2022-03-09') &
    (results_xgb_aci['date'] <= '2022-05-20')
].copy()

stress_zoom_xgb_aci['miss'] = (
    (stress_zoom_xgb_aci['actual'] < stress_zoom_xgb_aci['lower_bound']) |
    (stress_zoom_xgb_aci['actual'] > stress_zoom_xgb_aci['upper_bound'])
)

stress_zoom_xgb_aci['hit'] = ~stress_zoom_xgb_aci['miss']
misses_xgb_aci = stress_zoom_xgb_aci[stress_zoom_xgb_aci['miss']]

plt.figure(figsize=(12, 6))

plt.plot(
    stress_zoom_xgb_aci['date'],
    stress_zoom_xgb_aci['actual'],
    label='Actual',
    color="#1F77B4",
    linewidth=1.5
)

plt.plot(
    stress_zoom_xgb_aci['date'],
    stress_zoom_xgb_aci['prediction'],
    label='Predicted',
    color="#FF7F0E",
    linewidth=1.5,
    alpha=0.7
)

plt.fill_between(
    stress_zoom_xgb_aci['date'],
    stress_zoom_xgb_aci['lower_bound'],
    stress_zoom_xgb_aci['upper_bound'],
    color="#CFE3F2",
    alpha=0.35,
    label='Prediction interval'
)

plt.scatter(
    misses_xgb_aci['date'],
    misses_xgb_aci['actual'],
    s=50,
    color='red',
    label='Misses',
    alpha=1.0
)

plt.scatter(
    stress_zoom_xgb_aci[stress_zoom_xgb_aci['hit']]['date'],
    stress_zoom_xgb_aci[stress_zoom_xgb_aci['hit']]['actual'],
    s=50,
    color='green',
    label='Hits',
    alpha=0.7
)

plt.title('XGB + ACI: Stress period zoom')
plt.xlabel('Date')
plt.ylabel('Brent Price USD')
plt.grid(axis='y', alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# subplots  for ACI DT and RF

stress_start = "2022-03-09"
stress_end = "2022-05-20"

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True) # subplots witht same Y axis

plot_data = [
    ("Decision Tree + ACI", results_dt_aci, axes[0]), # plot 0
    ("Random Forest + ACI", results_rf_aci, axes[1])  # plot 1
]

for title, df, ax in plot_data: # loop results and axes

    stress_zoom = df[
        (df["date"] >= stress_start) &
        (df["date"] <= stress_end)
    ].copy()

    stress_zoom["miss"] = (
        (stress_zoom["actual"] < stress_zoom["lower_bound"]) |
        (stress_zoom["actual"] > stress_zoom["upper_bound"])
    )

    stress_zoom["hit"] = ~stress_zoom["miss"]

    misses = stress_zoom[stress_zoom["miss"]]
    hits = stress_zoom[stress_zoom["hit"]]

    ax.plot(
        stress_zoom["date"],
        stress_zoom["actual"],
        label="Actual",
        color="#1F77B4",
        linewidth=1.5
    )

    ax.plot(
        stress_zoom["date"],
        stress_zoom["prediction"],
        label="Predicted",
        color="#FF7F0E",
        linewidth=1.5,
        alpha=0.7
    )

    ax.fill_between(
        stress_zoom["date"],
        stress_zoom["lower_bound"],
        stress_zoom["upper_bound"],
        color="#CFE3F2",
        alpha=0.35,
        label="Prediction interval"
    )

    ax.scatter(
        misses["date"],
        misses["actual"],
        s=50,
        color="red",
        label="Misses",
        alpha=1.0
    )

    ax.scatter(
        hits["date"],
        hits["actual"],
        s=50,
        color="green",
        label="Hits",
        alpha=0.7
    )

    ax.set_title(f"{title}: Stress-period zoom")
    ax.set_xlabel("Date")
    ax.set_ylabel("Brent Price USD")
    ax.grid(axis="y", alpha=0.25)

handles, labels = axes[0].get_legend_handles_labels() # get legend from first plot
fig.legend(handles, labels, loc="upper center", ncol=5, frameon=True) # global legend

plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.show()

In [ ]:
# same for lgbm and aci

stress_start = "2022-03-09"
stress_end = "2022-05-20"

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

plot_data = [
    ("LGBM + ACI", results_lgbm_aci, axes[0]),
    ("XGB + ACI", results_xgb_aci, axes[1])
]

for title, df, ax in plot_data:

    stress_zoom = df[
        (df["date"] >= stress_start) &
        (df["date"] <= stress_end)
    ].copy()

    stress_zoom["miss"] = (
        (stress_zoom["actual"] < stress_zoom["lower_bound"]) |
        (stress_zoom["actual"] > stress_zoom["upper_bound"])
    )

    stress_zoom["hit"] = ~stress_zoom["miss"]

    misses = stress_zoom[stress_zoom["miss"]]
    hits = stress_zoom[stress_zoom["hit"]]

    ax.plot(
        stress_zoom["date"],
        stress_zoom["actual"],
        label="Actual",
        color="#1F77B4",
        linewidth=1.5
    )

    ax.plot(
        stress_zoom["date"],
        stress_zoom["prediction"],
        label="Predicted",
        color="#FF7F0E",
        linewidth=1.5,
        alpha=0.7
    )

    ax.fill_between(
        stress_zoom["date"],
        stress_zoom["lower_bound"],
        stress_zoom["upper_bound"],
        color="#CFE3F2",
        alpha=0.35,
        label="Prediction interval"
    )

    ax.scatter(
        misses["date"],
        misses["actual"],
        s=50,
        color="red",
        label="Misses",
        alpha=1.0
    )

    ax.scatter(
        hits["date"],
        hits["actual"],
        s=50,
        color="green",
        label="Hits",
        alpha=0.7
    )

    ax.set_title(f"{title}: Stress-period zoom")
    ax.set_xlabel("Date")
    ax.set_ylabel("Brent Price USD")
    ax.grid(axis="y", alpha=0.25)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=5, frameon=True)

plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.show()

In [ ]:
final_summary_all = pd.concat([
    summary_dt_cp_long,
    summary_dt_enbpi_long,
    summary_dt_aci_long,
    summary_rf_cp_long,
    summary_rf_enbpi_long,
    summary_rf_aci_long,
    summary_lgbm_cp_long,
    summary_lgbm_enbpi_long,
    summary_lgbm_aci_long,
    summary_xgb_cp_long,
    summary_xgb_enbpi_long,
    summary_xgb_aci_long
], ignore_index=True)

final_summary_all
# create a final summary table with all models and methods for the total period

In [ ]:
# make a dataframe  with only stress regimes

final_summary_stress = final_summary_all[final_summary_all['Regime'] == 'Stress'].copy()
final_summary_stress

In [ ]:
# make a dataframe with only non stress regimes
final_summary_non_stress = final_summary_all[final_summary_all['Regime'] == 'Non-Stress'].copy()
final_summary_non_stress

In [ ]:
# make a dataframe with only total regimes
final_summary_total = final_summary_all[final_summary_all['Regime'] == 'Total'].copy()
final_summary_total

In [ ]:
# combine a new DF with color codes for the methods and models to be used in the plots

model_order = ["Decision Tree", "Random Forest", "LGBM", "XGB"]
method_order = ["Baseline CP", "EnbPI", "ACI"]
regime_order = ["Total", "Stress", "Non-Stress"]
# set categorical order for models, methods and regimes so that they appear in the right order in the plots
final_summary_all["Model"] = pd.Categorical(final_summary_all["Model"], categories=model_order, ordered=True) 
final_summary_all["Method"] = pd.Categorical(final_summary_all["Method"], categories=method_order, ordered=True)
final_summary_all["Regime"] = pd.Categorical(final_summary_all["Regime"], categories=regime_order, ordered=True)

method_palette = {
    "Baseline CP": "#ADB5BD",
    "EnbPI": "#6C757D",
    "ACI": "#B56576"
}

edge_color = "#2F3E46"

In [ ]:
# # coverag width trade off plot of stress

stress_results = final_summary_all[final_summary_all["Regime"] == "Stress"].copy() # from above

plt.figure(figsize=(8, 5.5))

sns.scatterplot(
    data=stress_results,
    x="Mean Interval Width",
    y="Coverage",
    hue="Method", # create a symbol for each method
    style="Model", # create a symbol for each model
    palette=method_palette,
    s=120,
    edgecolor=edge_color,
    linewidth=0.7
)

plt.axhline(
    y=0.90,
    color=edge_color,
    linestyle="--",
    linewidth=1.2,
    label="Target coverage = 0.90"
)

plt.title("Coverage-Width Trade-off During Stress Periods")
plt.xlabel("Mean Interval Width")
plt.ylabel("Empirical Coverage")
plt.ylim(0, 1.0)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", alpha=0.12)

sns.despine()
plt.legend(frameon=True, bbox_to_anchor=(1.02, 1), loc="upper left") # legend outside the plot
plt.tight_layout() 
plt.show()

In [ ]:
# coverage width trade off plot for total period    
# same as above

stress_results = final_summary_all[final_summary_all["Regime"] == "Total"].copy()

plt.figure(figsize=(8, 5.5))

sns.scatterplot(
    data=stress_results,
    x="Mean Interval Width",
    y="Coverage",
    hue="Method", # symbols for methods
    style="Model",
    palette=method_palette,
    s=120,
    edgecolor=edge_color,
    linewidth=0.7
)

plt.axhline(
    y=0.90,
    color=edge_color,
    linestyle="--",
    linewidth=1.2,
    label="Target coverage = 0.90"
)

plt.title("Coverage-Width Trade-off For Total Period")
plt.xlabel("Mean Interval Width")
plt.ylabel("Empirical Coverage")
plt.ylim(0, 1.0)

plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", alpha=0.12)

sns.despine()
plt.legend(frameon=True, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
# non stress scatter plot
non_stress_results = final_summary_all[final_summary_all["Regime"] == "Non-Stress"].copy()
plt.figure(figsize=(8, 5.5))
sns.scatterplot(
    data=non_stress_results,
    x="Mean Interval Width",
    y="Coverage",
    hue="Method",
    style="Model",
    palette=method_palette,
    s=120,
    edgecolor=edge_color,
    linewidth=0.7
)
plt.axhline(
    y=0.90,
    color=edge_color,
    linestyle="--",
    linewidth=1.2,
    label="Target coverage = 0.90"
)
plt.title("Coverage-Width Trade-off During Non-Stress Periods")
plt.xlabel("Mean Interval Width")
plt.ylabel("Empirical Coverage")
plt.ylim(0, 1.0)
plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", alpha=0.12)
sns.despine()
plt.legend(frameon=True, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
# subplot of stress and non-stress coverage-width trade-off

fig, axes = plt.subplots(1, 2, figsize=(16, 6))  

# stress df
stress_results = final_summary_all[final_summary_all["Regime"] == "Stress"].copy()

sns.scatterplot(
    data=stress_results,
    x="Mean Interval Width",
    y="Coverage",
    hue="Method",
    style="Model",
    palette=method_palette,
    s=120,
    edgecolor=edge_color,
    linewidth=0.7,
    ax=axes[0]
)

axes[0].axhline(
    y=0.90,
    color=edge_color,
    linestyle="--",
    linewidth=1.2,
    label="Target coverage = 0.90"
)

axes[0].set_title("Coverage-Width Trade-off During Stress Periods")
axes[0].set_xlabel("Mean Interval Width")
axes[0].set_ylabel("Empirical Coverage")
axes[0].set_ylim(0, 1.0)
axes[0].grid(axis="y", alpha=0.25)
axes[0].grid(axis="x", alpha=0.12)
sns.despine(ax=axes[0])


# non stress df
non_stress_results = final_summary_all[final_summary_all["Regime"] == "Non-Stress"].copy()

sns.scatterplot(
    data=non_stress_results,
    x="Mean Interval Width",
    y="Coverage",
    hue="Method",
    style="Model",
    palette=method_palette,
    s=120,
    edgecolor=edge_color,
    linewidth=0.7,
    ax=axes[1]
)

axes[1].axhline(
    y=0.90,
    color=edge_color,
    linestyle="--",
    linewidth=1.2,
    label="Target coverage = 0.90"
)

axes[1].set_title("Coverage-Width Trade-off During Non-Stress Periods")
axes[1].set_xlabel("Mean Interval Width")
axes[1].set_ylabel("Empirical Coverage")
axes[1].set_ylim(0, 1.0)
axes[1].grid(axis="y", alpha=0.25)
axes[1].grid(axis="x", alpha=0.12)
sns.despine(ax=axes[1]) # remove top and right spines for both plots

axes[0].legend_.remove()
axes[1].legend(frameon=True, bbox_to_anchor=(1.02, 1), loc="upper left") # only show legend for the second plot

plt.tight_layout()
plt.show()

In [ ]:
# coverage width plot for total period

total_results = final_summary_all[final_summary_all["Regime"] == "Total"].copy() # create new df with total res
plt.figure(figsize=(7, 4.5))
sns.scatterplot(
    data=total_results,
    x="Mean Interval Width",
    y="Coverage",
    hue="Method",
    style="Model",
    palette=method_palette, # aligning colors
    s=120, # size of points
    edgecolor=edge_color,
    linewidth=0.7
)
plt.axhline(
    y=0.90,
    color=edge_color,
    linestyle="--",
    linewidth=1.2,
    label="Target coverage = 0.90"
)
plt.title("Coverage-Width Trade-off For Total Period")
plt.xlabel("Mean Interval Width")
plt.ylabel("Empirical Coverage")
plt.ylim(0, 1.0)
plt.grid(axis="y", alpha=0.25)
plt.grid(axis="x", alpha=0.12)
sns.despine()
plt.legend(frameon=True, bbox_to_anchor=(1.02, 1), loc="upper left") # place legend outside
plt.tight_layout()
plt.show()


In [ ]:
# coverage bar plot for stress and non-stress regimes

regime_results = final_summary_all[
    final_summary_all["Regime"].isin(["Stress", "Non-Stress"]) # filter for stress and non-stress regimes
].copy()

# sub plots side by side with sharey axis
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for ax, regime in zip(axes, ["Stress", "Non-Stress"]): 
    data = regime_results[regime_results["Regime"] == regime]
# stress
    sns.barplot(
        data=data,
        x="Model",
        y="Coverage",
        hue="Method",
        palette=method_palette,
        edgecolor=edge_color,
        linewidth=0.7,
        ax=ax
    )
# dashed for marking nominal coverage
    ax.axhline(
        y=0.90,
        color=edge_color,
        linestyle="--",
        linewidth=1.2
    )
#current regime 
    ax.set_title(f"Empirical Coverage: {regime}") # title with regime name
    ax.set_xlabel("")
    ax.set_ylabel("Empirical Coverage" if regime == "Stress" else "") # only show y label for the first plot
    ax.set_ylim(0, 1.0)
    ax.grid(axis="y", alpha=0.25)
    ax.grid(axis="x", visible=False)

    if regime == "Non-Stress": 
        ax.legend_.remove()

sns.despine() # for cleaner plots
handles, labels = axes[0].get_legend_handles_labels() # common legend for both plots
fig.legend(handles, labels, loc="upper center", ncol=4, frameon=True)
axes[0].legend_.remove()

plt.tight_layout(rect=[0, 0, 1, 0.90]) 
plt.show()

In [ ]:
# wiinkler score by regime: Stress vs Non-Stress

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True) # create subplots for stress and non-stress, share y-axis for better comparison

for ax, regime in zip(axes, ["Stress", "Non-Stress"]):
    data = regime_results[regime_results["Regime"] == regime]

    sns.barplot(
        data=data,
        x="Model",
        y="Winkler Score", # just change y variable to winkler score
        hue="Method",
        palette=method_palette,
        edgecolor=edge_color,
        linewidth=0.7,
        ax=ax
    )

    ax.set_title(f"Winkler Score: {regime}")
    ax.set_xlabel("")
    ax.set_ylabel("Winkler Score" if regime == "Stress" else "")
    ax.grid(axis="y", alpha=0.25)
    ax.grid(axis="x", visible=False)

    if regime == "Non-Stress":
        ax.legend_.remove()

sns.despine() # for cleaner plots
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=True) # place legend above the plots, ncol=3 for better spacing
axes[0].legend_.remove()

plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.show()

In [ ]:
# interval width by regime: Stress vs Non-Stress
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True) # create subplots with shared y-axis

for ax, regime in zip(axes, ["Stress", "Non-Stress"]):
    data = regime_results[regime_results["Regime"] == regime] 
# plot average interval width for each model and cp method
    sns.barplot(
        data=data,
        x="Model",
        y="Mean Interval Width", # change y variable to mean interval width
        hue="Method",
        palette=method_palette,
        edgecolor=edge_color,
        linewidth=0.7,
        ax=ax
    )

    ax.set_title(f"Mean Interval Width: {regime}")
    ax.set_xlabel("")
    ax.set_ylabel("Mean Interval Width" if regime == "Stress" else "")

    ax.grid(axis="y", alpha=0.25)
    ax.grid(axis="x", visible=False)

    if regime == "Non-Stress":
        ax.legend_.remove()

sns.despine() # for cleaner plots

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=True) 
axes[0].legend_.remove()

plt.tight_layout(rect=[0, 0, 1, 0.90]) # adjustments
plt.show()

In [ ]:
# 30-day rolling empirical coverage for all models and CP frameworks
# to inspect the adaptiveness of combined frameworks
rolling_window = 30 # to smoothen curves
model_results = [
    ("Decision Tree", results_dt_cp, results_dt_enbpi, results_dt_aci),
    ("Random Forest", results_rf_cp, results_rf_enbpi, results_rf_aci),
    ("LGBM", results_lgbm_cp, results_lgbm_enbpi, results_lgbm_aci),
    ("XGB", results_xgb_cp, results_xgb_enbpi, results_xgb_aci)
]

fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharey=True) # subplots for each model

for ax, (model_name, baseline_df, enbpi_df, aci_df) in zip(axes.flatten(), model_results): # loop through models and their results

    for method_name, df in [ # loop through methods for each model
        ("Baseline CP", baseline_df)
        ,("EnbPI", enbpi_df)
        ,("ACI", aci_df)
    ]:
        rolling_coverage = df["covered"].rolling(rolling_window).mean() # calculate rolling coverage

        sns.lineplot(
            x=df["date"]
            ,y=rolling_coverage
            ,label=method_name
            ,color=method_palette[method_name]
            ,linewidth=1.5
            ,ax=ax
        )

    ax.axhline(
        y=0.90
        ,color=edge_color
        ,linestyle="--"
        ,linewidth=1.2
        ,label="Target coverage = 0.90"
    )

    ax.set_title(f"{model_name}: 30-Trading-Day Rolling Empirical Coverage")
    ax.set_xlabel("Date")
    ax.set_ylabel("Rolling Coverage")
    ax.set_ylim(0, 1.05)
    ax.grid(axis="y", alpha=0.25)
    ax.grid(axis="x", alpha=0.08)
    sns.despine(ax=ax)

# legend on top
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, frameon=True)
plt.tight_layout(rect=[0, 0, 1, 0.95]) # adjust layout to make room for legend
plt.show()